# Kenya Watch — Machine Learning Component
### Report Category Classification for Corruption Reports in Kenya

**Project:** A Machine Learning-Based Web Platform for Anonymous Reporting and Tracking of
Corruption Cases in Kenya
**Student:** Nyangabi Benvictor Mwangi (155499) — BBIT 4C, Strathmore University
**Supervisor:** Ms. Zainabu Muti

---

## Purpose

This notebook reproduces the offline ML pipeline in the project repository (`ml/src/`) so the
complete workflow can be inspected and re-run in one place. It is organised to map directly onto
the assessment rubric:

| Rubric component | Section |
|---|---|
| Introduction & Problem Definition | **1** |
| Dataset & Preprocessing (description, EDA, feature engineering) | **2** |
| Methodology & Model Selection (choice, justification, implementation) | **3** |
| Model Evaluation (metrics, comparison, visualisation) | **4** |
| Deployment (integration strategy + demonstration) | **5** |

**To run:** `Runtime ▸ Run all`. No uploads or setup required.

## ⚠️ Honest scope statement (please read first)

The corpus used here is a **synthetic, hand-authored demonstration corpus** of fictional
reports — not real citizen submissions. It exists to exercise and validate the *pipeline*.
Therefore:

- The scores in Section 4 measure performance **on the synthetic corpus only**.
- They are **not** an estimate of accuracy on real Kenyan corruption reports.
- The live demo uses the same synthetic-only trained category model when its local artifact and
  Python environment are provisioned. If inference is unavailable, the app uses keyword category
  rules instead. Neither result is a finding; urgency remains keyword-based.
- The model has no calibrated probability or confidence score, and the synthetic examples do not
  establish real-world usefulness or safety.
- Using real reports requires the data-governance process documented in `ml/README.md`.


---
# 1. Introduction and Problem Definition
*(Rubric: 2 marks clarity of problem definition · 3 marks relevance and motivation)*

## 1.1 The problem

Corruption in Kenya is widespread, under-reported and rarely followed up. Citizens who witness or
experience bribery, embezzlement, procurement fraud or abuse of office are deterred by three things:

1. **Fear of retaliation** — reporting means becoming identifiable.
2. **No feedback** — complaints disappear, so trust in the process erodes.
3. **No prioritisation** — oversight bodies are overwhelmed, and serious cases are not triaged fast enough.

Kenya Watch addresses (1) with an anonymity-first architecture: anonymous reporters have **no
account at all**, and their identity is never collected, so there is nothing to leak. It addresses
(2) with a tracking reference + secret access key and a moderated agency-response loop.

This notebook addresses **(3) — triage**.

## 1.2 Where machine learning fits

A human review team cannot read and prioritise every narrative quickly. The platform therefore
uses lightweight text classification to **assist** triage:

- **Suggest a category** for each report (8 statutory categories + `Other`).
- **Signal urgency** so High/Critical cases escalate if unclaimed past a threshold.

**The ML layer is assistive, never decision-making:** it runs asynchronously after submission so it
can never block a report; it never sees or uses reporter identity (which is not collected for
anonymous reports); and category labels used for training come from a **double-blind human review**
workflow, not from the reporter's own selection.

## 1.3 Why this matters

Speed of first response is the single biggest driver of citizen trust in a reporting platform. A
reliable triage assist shortens time-to-first-review, surfaces severe cases early, and — because the
system also collects independent human labels — makes the machine layer **measurable and
accountable** rather than a black box.


---
# 2. Dataset and Preprocessing
*(Rubric: 3 marks dataset description · 4 marks preprocessing/feature engineering · 2 marks EDA)*

## 2.1 Dataset description

| Property | Value |
|---|---|
| Source | Deterministic, hand-authored **synthetic** corpus (`ml/src/generate_synthetic_dataset.py`) |
| Size | **540 rows** = 9 categories × 15 scenario groups × 4 variants |
| Variants per scenario | 2 English + 2 draft Kiswahili |
| Labels | The 8 statutory categories (Anti-Corruption and Economic Crimes Act) + `Other` |
| Incident grouping | Each fictional incident = one `group_id`; all variants of one incident stay in the same fold |
| Real personal data | **None** |

**Schema:** `text, label, group_id, language, county`

- `text` — the report narrative (the only model input besides metadata used for evaluation).
- `label` — independently assignable category (the target).
- `group_id` — the **incident** key. Critical for leakage-free validation (explained in §3.2).
- `language`, `county` — used **only** for aggregate subgroup evaluation, never as features.

**Why synthetic first?** Training on real reports requires institutional approval, de-identification
and reviewed labels. The synthetic corpus validates the pipeline's *mechanics* — grouping, nested
validation, metric computation — before any sensitive data is touched.

## 2.2 The corpus is embedded below

So that this notebook runs anywhere with **no file uploads**, the 540-row corpus is embedded as a
CSV string. It is byte-identical to `ml/data/synthetic/kenya_watch_synthetic_seed.csv` in the
repository.


In [ ]:
%pip -q install "scikit-learn>=1.5,<2" "joblib>=1.4,<2" pandas matplotlib


In [ ]:
import io, csv, json, statistics
from collections import Counter
from pathlib import Path
import pandas as pd

CATEGORIES = (
    "Bribery", "Embezzlement", "Procurement Fraud", "Abuse of Office",
    "Nepotism/Favoritism", "Extortion", "Fraud/Forgery of Documents",
    "Conflict of Interest", "Other",
)

CSV_DATA = 'text,label,group_id,language,county\nA licensing officer allegedly asked for an unofficial payment before approving a complete small-business permit application.,Bribery,synthetic-bribery-01,English,Nairobi\nThe report states that a licensing officer allegedly asked for an unofficial payment before approving a complete small-business permit application.,Bribery,synthetic-bribery-01,English,Nairobi\nAfisa wa leseni anadaiwa kuomba malipo yasiyo rasmi kabla ya kuidhinisha ombi kamili la kibali cha biashara ndogo.,Bribery,synthetic-bribery-01,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, afisa wa leseni anadaiwa kuomba malipo yasiyo rasmi kabla ya kuidhinisha ombi kamili la kibali cha biashara ndogo.",Bribery,synthetic-bribery-01,Kiswahili,Nairobi\nA resident says a public clerk requested cash to move a land-record search ahead of other applications.,Bribery,synthetic-bribery-02,English,Mombasa\nThe report states that a resident says a public clerk requested cash to move a land-record search ahead of other applications.,Bribery,synthetic-bribery-02,English,Mombasa\nMkazi anasema karani wa umma aliomba pesa ili kuharakisha upekuzi wa rekodi ya ardhi kuliko maombi mengine.,Bribery,synthetic-bribery-02,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, mkazi anasema karani wa umma aliomba pesa ili kuharakisha upekuzi wa rekodi ya ardhi kuliko maombi mengine.",Bribery,synthetic-bribery-02,Kiswahili,Mombasa\nA county employee allegedly offered to speed up inspection clearance in exchange for a personal payment.,Bribery,synthetic-bribery-03,English,Kisumu\nThe report states that a county employee allegedly offered to speed up inspection clearance in exchange for a personal payment.,Bribery,synthetic-bribery-03,English,Kisumu\nMfanyakazi wa kaunti anadaiwa kutoa ahadi ya kuharakisha kibali cha ukaguzi badala ya malipo binafsi.,Bribery,synthetic-bribery-03,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, mfanyakazi wa kaunti anadaiwa kutoa ahadi ya kuharakisha kibali cha ukaguzi badala ya malipo binafsi.",Bribery,synthetic-bribery-03,Kiswahili,Kisumu\nA public health inspector allegedly requested a gift before issuing a routine food-premises certificate.,Bribery,synthetic-bribery-04,English,Nakuru\nThe report states that a public health inspector allegedly requested a gift before issuing a routine food-premises certificate.,Bribery,synthetic-bribery-04,English,Nakuru\nMkaguzi wa afya ya umma anadaiwa kuomba zawadi kabla ya kutoa cheti cha kawaida cha eneo la chakula.,Bribery,synthetic-bribery-04,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, mkaguzi wa afya ya umma anadaiwa kuomba zawadi kabla ya kutoa cheti cha kawaida cha eneo la chakula.",Bribery,synthetic-bribery-04,Kiswahili,Nakuru\nA service applicant reports that an official asked for money to release a document that was already approved.,Bribery,synthetic-bribery-05,English,Turkana\nThe report states that a service applicant reports that an official asked for money to release a document that was already approved.,Bribery,synthetic-bribery-05,English,Turkana\nMwombaji wa huduma anaripoti kuwa afisa aliomba pesa ili kutoa hati ambayo tayari ilikuwa imeidhinishwa.,Bribery,synthetic-bribery-05,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, mwombaji wa huduma anaripoti kuwa afisa aliomba pesa ili kutoa hati ambayo tayari ilikuwa imeidhinishwa.",Bribery,synthetic-bribery-05,Kiswahili,Turkana\nA public employee allegedly asked a supplier for a personal benefit in return for processing an invoice promptly.,Bribery,synthetic-bribery-06,English,Garissa\nThe report states that a public employee allegedly asked a supplier for a personal benefit in return for processing an invoice promptly.,Bribery,synthetic-bribery-06,English,Garissa\nMfanyakazi wa umma anadaiwa kuomba manufaa binafsi kutoka kwa msambazaji ili kushughulikia ankara haraka.,Bribery,synthetic-bribery-06,Kiswahili,Garissa\n"Kwa mujibu wa taarifa, mfanyakazi wa umma anadaiwa kuomba manufaa binafsi kutoka kwa msambazaji ili kushughulikia ankara haraka.",Bribery,synthetic-bribery-06,Kiswahili,Garissa\nA resident alleges that an unofficial fee was requested to obtain a place in a public-service queue.,Bribery,synthetic-bribery-07,English,Kilifi\nThe report states that a resident alleges that an unofficial fee was requested to obtain a place in a public-service queue.,Bribery,synthetic-bribery-07,English,Kilifi\nMkazi anadai kuwa ada isiyo rasmi iliombwa ili kupata nafasi kwenye foleni ya huduma ya umma.,Bribery,synthetic-bribery-07,Kiswahili,Kilifi\n"Kwa mujibu wa taarifa, mkazi anadai kuwa ada isiyo rasmi iliombwa ili kupata nafasi kwenye foleni ya huduma ya umma.",Bribery,synthetic-bribery-07,Kiswahili,Kilifi\nAn officer allegedly promised to omit a routine penalty after receiving a private payment.,Bribery,synthetic-bribery-08,English,Bungoma\nThe report states that an officer allegedly promised to omit a routine penalty after receiving a private payment.,Bribery,synthetic-bribery-08,English,Bungoma\nAfisa anadaiwa kuahidi kuondoa adhabu ya kawaida baada ya kupokea malipo ya faragha.,Bribery,synthetic-bribery-08,Kiswahili,Bungoma\n"Kwa mujibu wa taarifa, afisa anadaiwa kuahidi kuondoa adhabu ya kawaida baada ya kupokea malipo ya faragha.",Bribery,synthetic-bribery-08,Kiswahili,Bungoma\nA contractor says a public employee requested a personal commission for certifying completed work.,Bribery,synthetic-bribery-09,English,Meru\nThe report states that a contractor says a public employee requested a personal commission for certifying completed work.,Bribery,synthetic-bribery-09,English,Meru\nMkandarasi anasema mfanyakazi wa umma aliomba kamisheni binafsi ili kuthibitisha kazi iliyokamilika.,Bribery,synthetic-bribery-09,Kiswahili,Meru\n"Kwa mujibu wa taarifa, mkandarasi anasema mfanyakazi wa umma aliomba kamisheni binafsi ili kuthibitisha kazi iliyokamilika.",Bribery,synthetic-bribery-09,Kiswahili,Meru\nA trader reports being asked for an unofficial payment before a market-stall allocation was processed.,Bribery,synthetic-bribery-10,English,Kitui\nThe report states that a trader reports being asked for an unofficial payment before a market-stall allocation was processed.,Bribery,synthetic-bribery-10,English,Kitui\nMfanyabiashara anaripoti kuombwa malipo yasiyo rasmi kabla ya kushughulikiwa kwa mgao wa kibanda sokoni.,Bribery,synthetic-bribery-10,Kiswahili,Kitui\n"Kwa mujibu wa taarifa, mfanyabiashara anaripoti kuombwa malipo yasiyo rasmi kabla ya kushughulikiwa kwa mgao wa kibanda sokoni.",Bribery,synthetic-bribery-10,Kiswahili,Kitui\nA public clerk allegedly requested money to issue a routine copy of a record already held by the office.,Bribery,synthetic-bribery-11,English,Nairobi\nThe report states that a public clerk allegedly requested money to issue a routine copy of a record already held by the office.,Bribery,synthetic-bribery-11,English,Nairobi\nKarani wa umma anadaiwa kuomba pesa ili kutoa nakala ya kawaida ya rekodi ambayo ofisi tayari inayo.,Bribery,synthetic-bribery-11,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, karani wa umma anadaiwa kuomba pesa ili kutoa nakala ya kawaida ya rekodi ambayo ofisi tayari inayo.",Bribery,synthetic-bribery-11,Kiswahili,Nairobi\nA supplier reports an official asked for a personal reward before confirming delivery of standard materials.,Bribery,synthetic-bribery-12,English,Mombasa\nThe report states that a supplier reports an official asked for a personal reward before confirming delivery of standard materials.,Bribery,synthetic-bribery-12,English,Mombasa\nMsambazaji anaripoti kuwa afisa aliomba zawadi binafsi kabla ya kuthibitisha uwasilishaji wa vifaa vya kawaida.,Bribery,synthetic-bribery-12,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, msambazaji anaripoti kuwa afisa aliomba zawadi binafsi kabla ya kuthibitisha uwasilishaji wa vifaa vya kawaida.",Bribery,synthetic-bribery-12,Kiswahili,Mombasa\nA resident says a public officer requested a private payment to schedule an ordinary inspection.,Bribery,synthetic-bribery-13,English,Kisumu\nThe report states that a resident says a public officer requested a private payment to schedule an ordinary inspection.,Bribery,synthetic-bribery-13,English,Kisumu\nMkazi anasema afisa wa umma aliomba malipo ya faragha ili kupanga ukaguzi wa kawaida.,Bribery,synthetic-bribery-13,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, mkazi anasema afisa wa umma aliomba malipo ya faragha ili kupanga ukaguzi wa kawaida.",Bribery,synthetic-bribery-13,Kiswahili,Kisumu\nAn applicant alleges a staff member requested a personal gift before forwarding a complete service form.,Bribery,synthetic-bribery-14,English,Nakuru\nThe report states that an applicant alleges a staff member requested a personal gift before forwarding a complete service form.,Bribery,synthetic-bribery-14,English,Nakuru\nMwombaji anadai kuwa mfanyakazi aliomba zawadi binafsi kabla ya kupeleka fomu kamili ya huduma.,Bribery,synthetic-bribery-14,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, mwombaji anadai kuwa mfanyakazi aliomba zawadi binafsi kabla ya kupeleka fomu kamili ya huduma.",Bribery,synthetic-bribery-14,Kiswahili,Nakuru\nA business owner says a public officer asked for money to perform a duty that should have been routine.,Bribery,synthetic-bribery-15,English,Turkana\nThe report states that a business owner says a public officer asked for money to perform a duty that should have been routine.,Bribery,synthetic-bribery-15,English,Turkana\nMmiliki wa biashara anasema afisa wa umma aliomba pesa ili kutekeleza jukumu la kawaida.,Bribery,synthetic-bribery-15,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, mmiliki wa biashara anasema afisa wa umma aliomba pesa ili kutekeleza jukumu la kawaida.",Bribery,synthetic-bribery-15,Kiswahili,Turkana\nA school committee says money allocated for classroom repairs was withdrawn but the repairs were never done.,Embezzlement,synthetic-embezzlement-01,English,Nairobi\nThe report states that a school committee says money allocated for classroom repairs was withdrawn but the repairs were never done.,Embezzlement,synthetic-embezzlement-01,English,Nairobi\nKamati ya shule inasema pesa za ukarabati wa darasa zilitolewa lakini ukarabati haukufanyika.,Embezzlement,synthetic-embezzlement-01,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, kamati ya shule inasema pesa za ukarabati wa darasa zilitolewa lakini ukarabati haukufanyika.",Embezzlement,synthetic-embezzlement-01,Kiswahili,Nairobi\nA clinic inventory review allegedly found public medical supplies recorded as issued but missing from storage.,Embezzlement,synthetic-embezzlement-02,English,Mombasa\nThe report states that a clinic inventory review allegedly found public medical supplies recorded as issued but missing from storage.,Embezzlement,synthetic-embezzlement-02,English,Mombasa\nUkaguzi wa stoo ya kliniki unadaiwa kupata vifaa vya afya vya umma vilivyoandikwa kuwa vimetolewa lakini havipo.,Embezzlement,synthetic-embezzlement-02,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, ukaguzi wa stoo ya kliniki unadaiwa kupata vifaa vya afya vya umma vilivyoandikwa kuwa vimetolewa lakini havipo.",Embezzlement,synthetic-embezzlement-02,Kiswahili,Mombasa\nA community group reports that funds for a water point were transferred to an unrelated private account.,Embezzlement,synthetic-embezzlement-03,English,Kisumu\nThe report states that a community group reports that funds for a water point were transferred to an unrelated private account.,Embezzlement,synthetic-embezzlement-03,English,Kisumu\nKikundi cha jamii kinaripoti kuwa pesa za kituo cha maji zilihamishiwa kwenye akaunti binafsi isiyohusika.,Embezzlement,synthetic-embezzlement-03,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, kikundi cha jamii kinaripoti kuwa pesa za kituo cha maji zilihamishiwa kwenye akaunti binafsi isiyohusika.",Embezzlement,synthetic-embezzlement-03,Kiswahili,Kisumu\nA public works team says fuel bought for government vehicles was diverted for private use.,Embezzlement,synthetic-embezzlement-04,English,Nakuru\nThe report states that a public works team says fuel bought for government vehicles was diverted for private use.,Embezzlement,synthetic-embezzlement-04,English,Nakuru\nTimu ya kazi za umma inasema mafuta yaliyonunuliwa kwa magari ya serikali yalielekezwa kwa matumizi binafsi.,Embezzlement,synthetic-embezzlement-04,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, timu ya kazi za umma inasema mafuta yaliyonunuliwa kwa magari ya serikali yalielekezwa kwa matumizi binafsi.",Embezzlement,synthetic-embezzlement-04,Kiswahili,Nakuru\nA project audit allegedly found that building materials paid for from a public budget were removed for private use.,Embezzlement,synthetic-embezzlement-05,English,Turkana\nThe report states that a project audit allegedly found that building materials paid for from a public budget were removed for private use.,Embezzlement,synthetic-embezzlement-05,English,Turkana\nUkaguzi wa mradi unadaiwa kubaini kuwa vifaa vya ujenzi vilivyolipiwa na bajeti ya umma vilichukuliwa kwa matumizi binafsi.,Embezzlement,synthetic-embezzlement-05,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, ukaguzi wa mradi unadaiwa kubaini kuwa vifaa vya ujenzi vilivyolipiwa na bajeti ya umma vilichukuliwa kwa matumizi binafsi.",Embezzlement,synthetic-embezzlement-05,Kiswahili,Turkana\nA residents\' committee says collected public-service revenue was not deposited into the designated account.,Embezzlement,synthetic-embezzlement-06,English,Garissa\nThe report states that a residents\' committee says collected public-service revenue was not deposited into the designated account.,Embezzlement,synthetic-embezzlement-06,English,Garissa\nKamati ya wakazi inasema mapato ya huduma ya umma yaliyokusanywa hayakuwekwa kwenye akaunti iliyotengwa.,Embezzlement,synthetic-embezzlement-06,Kiswahili,Garissa\n"Kwa mujibu wa taarifa, kamati ya wakazi inasema mapato ya huduma ya umma yaliyokusanywa hayakuwekwa kwenye akaunti iliyotengwa.",Embezzlement,synthetic-embezzlement-06,Kiswahili,Garissa\n"Funds recorded for a drainage project were allegedly withdrawn, while the site remained untouched.",Embezzlement,synthetic-embezzlement-07,English,Kilifi\n"The report states that funds recorded for a drainage project were allegedly withdrawn, while the site remained untouched.",Embezzlement,synthetic-embezzlement-07,English,Kilifi\nPesa zilizoandikwa kwa mradi wa mifereji ya maji zinadaiwa kutolewa huku eneo la kazi likibaki bila shughuli.,Embezzlement,synthetic-embezzlement-07,Kiswahili,Kilifi\n"Kwa mujibu wa taarifa, pesa zilizoandikwa kwa mradi wa mifereji ya maji zinadaiwa kutolewa huku eneo la kazi likibaki bila shughuli.",Embezzlement,synthetic-embezzlement-07,Kiswahili,Kilifi\nA department report says equipment entrusted to a public office was sold without authorization.,Embezzlement,synthetic-embezzlement-08,English,Bungoma\nThe report states that a department report says equipment entrusted to a public office was sold without authorization.,Embezzlement,synthetic-embezzlement-08,English,Bungoma\nRipoti ya idara inasema vifaa vilivyokabidhiwa kwa ofisi ya umma viliuzwa bila idhini.,Embezzlement,synthetic-embezzlement-08,Kiswahili,Bungoma\n"Kwa mujibu wa taarifa, ripoti ya idara inasema vifaa vilivyokabidhiwa kwa ofisi ya umma viliuzwa bila idhini.",Embezzlement,synthetic-embezzlement-08,Kiswahili,Bungoma\nA local committee alleges that relief supplies received for public distribution were diverted before reaching recipients.,Embezzlement,synthetic-embezzlement-09,English,Meru\nThe report states that a local committee alleges that relief supplies received for public distribution were diverted before reaching recipients.,Embezzlement,synthetic-embezzlement-09,English,Meru\nKamati ya eneo inadai kuwa bidhaa za msaada zilizopokelewa kwa ugawaji wa umma zilielekezwa kwingine kabla ya kuwafikia walengwa.,Embezzlement,synthetic-embezzlement-09,Kiswahili,Meru\n"Kwa mujibu wa taarifa, kamati ya eneo inadai kuwa bidhaa za msaada zilizopokelewa kwa ugawaji wa umma zilielekezwa kwingine kabla ya kuwafikia walengwa.",Embezzlement,synthetic-embezzlement-09,Kiswahili,Meru\nAn audit note says a public account was charged for maintenance work that was not carried out.,Embezzlement,synthetic-embezzlement-10,English,Kitui\nThe report states that an audit note says a public account was charged for maintenance work that was not carried out.,Embezzlement,synthetic-embezzlement-10,English,Kitui\nDokezo la ukaguzi linasema akaunti ya umma ilitozwa kwa kazi ya matengenezo ambayo haikufanywa.,Embezzlement,synthetic-embezzlement-10,Kiswahili,Kitui\n"Kwa mujibu wa taarifa, dokezo la ukaguzi linasema akaunti ya umma ilitozwa kwa kazi ya matengenezo ambayo haikufanywa.",Embezzlement,synthetic-embezzlement-10,Kiswahili,Kitui\nA public office allegedly recorded salary payments for temporary workers who did not perform the listed work.,Embezzlement,synthetic-embezzlement-11,English,Nairobi\nThe report states that a public office allegedly recorded salary payments for temporary workers who did not perform the listed work.,Embezzlement,synthetic-embezzlement-11,English,Nairobi\nOfisi ya umma inadaiwa kurekodi malipo ya wafanyakazi wa muda ambao hawakufanya kazi iliyoandikwa.,Embezzlement,synthetic-embezzlement-11,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, ofisi ya umma inadaiwa kurekodi malipo ya wafanyakazi wa muda ambao hawakufanya kazi iliyoandikwa.",Embezzlement,synthetic-embezzlement-11,Kiswahili,Nairobi\nA committee reports that construction supplies purchased for a public facility were taken to a private site.,Embezzlement,synthetic-embezzlement-12,English,Mombasa\nThe report states that a committee reports that construction supplies purchased for a public facility were taken to a private site.,Embezzlement,synthetic-embezzlement-12,English,Mombasa\nKamati inaripoti kuwa vifaa vya ujenzi vilivyonunuliwa kwa kituo cha umma vilipelekwa kwenye eneo binafsi.,Embezzlement,synthetic-embezzlement-12,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, kamati inaripoti kuwa vifaa vya ujenzi vilivyonunuliwa kwa kituo cha umma vilipelekwa kwenye eneo binafsi.",Embezzlement,synthetic-embezzlement-12,Kiswahili,Mombasa\nMoney collected by a public unit allegedly disappeared before being entered into the official ledger.,Embezzlement,synthetic-embezzlement-13,English,Kisumu\nThe report states that money collected by a public unit allegedly disappeared before being entered into the official ledger.,Embezzlement,synthetic-embezzlement-13,English,Kisumu\nPesa zilizokusanywa na kitengo cha umma zinadaiwa kutoweka kabla ya kuingizwa kwenye daftari rasmi.,Embezzlement,synthetic-embezzlement-13,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, pesa zilizokusanywa na kitengo cha umma zinadaiwa kutoweka kabla ya kuingizwa kwenye daftari rasmi.",Embezzlement,synthetic-embezzlement-13,Kiswahili,Kisumu\nA review found that budgeted funds for a public water repair were used for an undocumented purpose.,Embezzlement,synthetic-embezzlement-14,English,Nakuru\nThe report states that a review found that budgeted funds for a public water repair were used for an undocumented purpose.,Embezzlement,synthetic-embezzlement-14,English,Nakuru\nUkaguzi ulibaini kuwa pesa zilizotengwa kwa ukarabati wa maji ya umma zilitumika kwa kusudi lisiloandikwa.,Embezzlement,synthetic-embezzlement-14,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, ukaguzi ulibaini kuwa pesa zilizotengwa kwa ukarabati wa maji ya umma zilitumika kwa kusudi lisiloandikwa.",Embezzlement,synthetic-embezzlement-14,Kiswahili,Nakuru\nA storekeeper allegedly removed public tools entrusted to the department and kept them for personal use.,Embezzlement,synthetic-embezzlement-15,English,Turkana\nThe report states that a storekeeper allegedly removed public tools entrusted to the department and kept them for personal use.,Embezzlement,synthetic-embezzlement-15,English,Turkana\nMtunza ghala anadaiwa kutoa zana za umma zilizokabidhiwa idara na kuzitumia binafsi.,Embezzlement,synthetic-embezzlement-15,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, mtunza ghala anadaiwa kutoa zana za umma zilizokabidhiwa idara na kuzitumia binafsi.",Embezzlement,synthetic-embezzlement-15,Kiswahili,Turkana\nSeveral suppliers allege that tender requirements were written to match one bidder\'s unique product.,Procurement Fraud,synthetic-procurement-fraud-01,English,Nairobi\nThe report states that several suppliers allege that tender requirements were written to match one bidder\'s unique product.,Procurement Fraud,synthetic-procurement-fraud-01,English,Nairobi\nWasambazaji kadhaa wanadai kuwa masharti ya zabuni yaliandikwa ili yafanane na bidhaa ya mzabuni mmoja.,Procurement Fraud,synthetic-procurement-fraud-01,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, wasambazaji kadhaa wanadai kuwa masharti ya zabuni yaliandikwa ili yafanane na bidhaa ya mzabuni mmoja.",Procurement Fraud,synthetic-procurement-fraud-01,Kiswahili,Nairobi\nA bid review allegedly found that competing offers used the same unusual wording and price pattern.,Procurement Fraud,synthetic-procurement-fraud-02,English,Mombasa\nThe report states that a bid review allegedly found that competing offers used the same unusual wording and price pattern.,Procurement Fraud,synthetic-procurement-fraud-02,English,Mombasa\nUkaguzi wa zabuni unadaiwa kubaini kuwa ofa pinzani zilitumia maneno yasiyo ya kawaida na mpangilio wa bei unaofanana.,Procurement Fraud,synthetic-procurement-fraud-02,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, ukaguzi wa zabuni unadaiwa kubaini kuwa ofa pinzani zilitumia maneno yasiyo ya kawaida na mpangilio wa bei unaofanana.",Procurement Fraud,synthetic-procurement-fraud-02,Kiswahili,Mombasa\nA public tender was allegedly awarded before the advertised deadline for submitting applications.,Procurement Fraud,synthetic-procurement-fraud-03,English,Kisumu\nThe report states that a public tender was allegedly awarded before the advertised deadline for submitting applications.,Procurement Fraud,synthetic-procurement-fraud-03,English,Kisumu\nZabuni ya umma inadaiwa kutolewa kabla ya tarehe ya mwisho iliyotangazwa ya kutuma maombi.,Procurement Fraud,synthetic-procurement-fraud-03,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, zabuni ya umma inadaiwa kutolewa kabla ya tarehe ya mwisho iliyotangazwa ya kutuma maombi.",Procurement Fraud,synthetic-procurement-fraud-03,Kiswahili,Kisumu\nA supplier says an evaluation panel changed scoring criteria after seeing the submitted bids.,Procurement Fraud,synthetic-procurement-fraud-04,English,Nakuru\nThe report states that a supplier says an evaluation panel changed scoring criteria after seeing the submitted bids.,Procurement Fraud,synthetic-procurement-fraud-04,English,Nakuru\nMsambazaji anasema jopo la tathmini lilibadilisha vigezo vya alama baada ya kuona zabuni zilizowasilishwa.,Procurement Fraud,synthetic-procurement-fraud-04,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, msambazaji anasema jopo la tathmini lilibadilisha vigezo vya alama baada ya kuona zabuni zilizowasilishwa.",Procurement Fraud,synthetic-procurement-fraud-04,Kiswahili,Nakuru\nA procurement notice allegedly omitted a required qualification in a way that excluded eligible local bidders.,Procurement Fraud,synthetic-procurement-fraud-05,English,Turkana\nThe report states that a procurement notice allegedly omitted a required qualification in a way that excluded eligible local bidders.,Procurement Fraud,synthetic-procurement-fraud-05,English,Turkana\nTangazo la ununuzi linadaiwa kuacha sifa iliyohitajika kwa namna iliyowazuia wazabuni wa eneo waliostahili.,Procurement Fraud,synthetic-procurement-fraud-05,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, tangazo la ununuzi linadaiwa kuacha sifa iliyohitajika kwa namna iliyowazuia wazabuni wa eneo waliostahili.",Procurement Fraud,synthetic-procurement-fraud-05,Kiswahili,Turkana\nA review alleges that a contract was split into smaller purchases to avoid the required open tender process.,Procurement Fraud,synthetic-procurement-fraud-06,English,Garissa\nThe report states that a review alleges that a contract was split into smaller purchases to avoid the required open tender process.,Procurement Fraud,synthetic-procurement-fraud-06,English,Garissa\nUkaguzi unadai kuwa mkataba uligawanywa kuwa manunuzi madogo ili kuepuka utaratibu wa zabuni ya wazi.,Procurement Fraud,synthetic-procurement-fraud-06,Kiswahili,Garissa\n"Kwa mujibu wa taarifa, ukaguzi unadai kuwa mkataba uligawanywa kuwa manunuzi madogo ili kuepuka utaratibu wa zabuni ya wazi.",Procurement Fraud,synthetic-procurement-fraud-06,Kiswahili,Garissa\nTwo bidders allegedly coordinated their prices and took turns submitting the lowest offer.,Procurement Fraud,synthetic-procurement-fraud-07,English,Kilifi\nThe report states that two bidders allegedly coordinated their prices and took turns submitting the lowest offer.,Procurement Fraud,synthetic-procurement-fraud-07,English,Kilifi\nWazabuni wawili wanadaiwa kuratibu bei zao na kubadilishana zamu ya kutoa ofa ya chini zaidi.,Procurement Fraud,synthetic-procurement-fraud-07,Kiswahili,Kilifi\n"Kwa mujibu wa taarifa, wazabuni wawili wanadaiwa kuratibu bei zao na kubadilishana zamu ya kutoa ofa ya chini zaidi.",Procurement Fraud,synthetic-procurement-fraud-07,Kiswahili,Kilifi\nA tender committee allegedly accepted a bid after its submission deadline while rejecting earlier compliant bids.,Procurement Fraud,synthetic-procurement-fraud-08,English,Bungoma\nThe report states that a tender committee allegedly accepted a bid after its submission deadline while rejecting earlier compliant bids.,Procurement Fraud,synthetic-procurement-fraud-08,English,Bungoma\nKamati ya zabuni inadaiwa kukubali zabuni baada ya muda wa mwisho huku ikikataa zabuni zilizotii masharti mapema.,Procurement Fraud,synthetic-procurement-fraud-08,Kiswahili,Bungoma\n"Kwa mujibu wa taarifa, kamati ya zabuni inadaiwa kukubali zabuni baada ya muda wa mwisho huku ikikataa zabuni zilizotii masharti mapema.",Procurement Fraud,synthetic-procurement-fraud-08,Kiswahili,Bungoma\nA public contract was allegedly awarded to a supplier that did not meet the published technical requirements.,Procurement Fraud,synthetic-procurement-fraud-09,English,Meru\nThe report states that a public contract was allegedly awarded to a supplier that did not meet the published technical requirements.,Procurement Fraud,synthetic-procurement-fraud-09,English,Meru\nMkataba wa umma unadaiwa kutolewa kwa msambazaji ambaye hakutimiza mahitaji ya kiufundi yaliyochapishwa.,Procurement Fraud,synthetic-procurement-fraud-09,Kiswahili,Meru\n"Kwa mujibu wa taarifa, mkataba wa umma unadaiwa kutolewa kwa msambazaji ambaye hakutimiza mahitaji ya kiufundi yaliyochapishwa.",Procurement Fraud,synthetic-procurement-fraud-09,Kiswahili,Meru\nA procurement file allegedly shows that the preferred bidder helped draft the specifications used to evaluate competitors.,Procurement Fraud,synthetic-procurement-fraud-10,English,Kitui\nThe report states that a procurement file allegedly shows that the preferred bidder helped draft the specifications used to evaluate competitors.,Procurement Fraud,synthetic-procurement-fraud-10,English,Kitui\nFaili ya ununuzi inadaiwa kuonyesha kuwa mzabuni aliyependekezwa alisaidia kuandika vipimo vilivyotumika kutathmini washindani.,Procurement Fraud,synthetic-procurement-fraud-10,Kiswahili,Kitui\n"Kwa mujibu wa taarifa, faili ya ununuzi inadaiwa kuonyesha kuwa mzabuni aliyependekezwa alisaidia kuandika vipimo vilivyotumika kutathmini washindani.",Procurement Fraud,synthetic-procurement-fraud-10,Kiswahili,Kitui\nA bid opening record allegedly lists prices that differ from the sealed offers delivered by suppliers.,Procurement Fraud,synthetic-procurement-fraud-11,English,Nairobi\nThe report states that a bid opening record allegedly lists prices that differ from the sealed offers delivered by suppliers.,Procurement Fraud,synthetic-procurement-fraud-11,English,Nairobi\nRekodi ya kufungua zabuni inadaiwa kuonyesha bei zinazotofautiana na ofa zilizofungwa zilizowasilishwa na wasambazaji.,Procurement Fraud,synthetic-procurement-fraud-11,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, rekodi ya kufungua zabuni inadaiwa kuonyesha bei zinazotofautiana na ofa zilizofungwa zilizowasilishwa na wasambazaji.",Procurement Fraud,synthetic-procurement-fraud-11,Kiswahili,Nairobi\n"A committee reportedly changed the tender scoring after bids were submitted, without issuing a public amendment.",Procurement Fraud,synthetic-procurement-fraud-12,English,Mombasa\n"The report states that a committee reportedly changed the tender scoring after bids were submitted, without issuing a public amendment.",Procurement Fraud,synthetic-procurement-fraud-12,English,Mombasa\nKamati inaripotiwa kubadilisha alama za zabuni baada ya kuwasilishwa bila kutoa marekebisho ya umma.,Procurement Fraud,synthetic-procurement-fraud-12,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, kamati inaripotiwa kubadilisha alama za zabuni baada ya kuwasilishwa bila kutoa marekebisho ya umma.",Procurement Fraud,synthetic-procurement-fraud-12,Kiswahili,Mombasa\nA contract review alleges that the winning supplier lacked a mandatory document required in the tender notice.,Procurement Fraud,synthetic-procurement-fraud-13,English,Kisumu\nThe report states that a contract review alleges that the winning supplier lacked a mandatory document required in the tender notice.,Procurement Fraud,synthetic-procurement-fraud-13,English,Kisumu\nUkaguzi wa mkataba unadai kuwa msambazaji mshindi hakuwa na hati ya lazima iliyotajwa kwenye tangazo la zabuni.,Procurement Fraud,synthetic-procurement-fraud-13,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, ukaguzi wa mkataba unadai kuwa msambazaji mshindi hakuwa na hati ya lazima iliyotajwa kwenye tangazo la zabuni.",Procurement Fraud,synthetic-procurement-fraud-13,Kiswahili,Kisumu\nA public entity allegedly gave one bidder advance access to tender information withheld from other participants.,Procurement Fraud,synthetic-procurement-fraud-14,English,Nakuru\nThe report states that a public entity allegedly gave one bidder advance access to tender information withheld from other participants.,Procurement Fraud,synthetic-procurement-fraud-14,English,Nakuru\nTaasisi ya umma inadaiwa kumpa mzabuni mmoja taarifa za zabuni mapema huku ikiwanyima washiriki wengine.,Procurement Fraud,synthetic-procurement-fraud-14,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, taasisi ya umma inadaiwa kumpa mzabuni mmoja taarifa za zabuni mapema huku ikiwanyima washiriki wengine.",Procurement Fraud,synthetic-procurement-fraud-14,Kiswahili,Nakuru\nAn evaluation panel allegedly discarded a compliant low bid using a reason not found in the published criteria.,Procurement Fraud,synthetic-procurement-fraud-15,English,Turkana\nThe report states that an evaluation panel allegedly discarded a compliant low bid using a reason not found in the published criteria.,Procurement Fraud,synthetic-procurement-fraud-15,English,Turkana\nJopo la tathmini linadaiwa kuondoa zabuni ya chini iliyotii masharti kwa sababu isiyokuwapo kwenye vigezo vilivyochapishwa.,Procurement Fraud,synthetic-procurement-fraud-15,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, jopo la tathmini linadaiwa kuondoa zabuni ya chini iliyotii masharti kwa sababu isiyokuwapo kwenye vigezo vilivyochapishwa.",Procurement Fraud,synthetic-procurement-fraud-15,Kiswahili,Turkana\nAn officer allegedly used official authority to stop a routine service for a resident after a personal dispute.,Abuse of Office,synthetic-abuse-of-office-01,English,Nairobi\nThe report states that an officer allegedly used official authority to stop a routine service for a resident after a personal dispute.,Abuse of Office,synthetic-abuse-of-office-01,English,Nairobi\nAfisa anadaiwa kutumia mamlaka rasmi kusimamisha huduma ya kawaida kwa mkazi baada ya mzozo binafsi.,Abuse of Office,synthetic-abuse-of-office-01,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, afisa anadaiwa kutumia mamlaka rasmi kusimamisha huduma ya kawaida kwa mkazi baada ya mzozo binafsi.",Abuse of Office,synthetic-abuse-of-office-01,Kiswahili,Nairobi\nA public manager allegedly ordered staff to use government equipment for a private event.,Abuse of Office,synthetic-abuse-of-office-02,English,Mombasa\nThe report states that a public manager allegedly ordered staff to use government equipment for a private event.,Abuse of Office,synthetic-abuse-of-office-02,English,Mombasa\nMsimamizi wa umma anadaiwa kuwaamuru wafanyakazi kutumia vifaa vya serikali kwa tukio binafsi.,Abuse of Office,synthetic-abuse-of-office-02,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, msimamizi wa umma anadaiwa kuwaamuru wafanyakazi kutumia vifaa vya serikali kwa tukio binafsi.",Abuse of Office,synthetic-abuse-of-office-02,Kiswahili,Mombasa\nA licensing supervisor allegedly bypassed the normal process to close a competitor\'s lawful business.,Abuse of Office,synthetic-abuse-of-office-03,English,Kisumu\nThe report states that a licensing supervisor allegedly bypassed the normal process to close a competitor\'s lawful business.,Abuse of Office,synthetic-abuse-of-office-03,English,Kisumu\nMsimamizi wa leseni anadaiwa kupuuza utaratibu wa kawaida ili kufunga biashara halali ya mshindani.,Abuse of Office,synthetic-abuse-of-office-03,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, msimamizi wa leseni anadaiwa kupuuza utaratibu wa kawaida ili kufunga biashara halali ya mshindani.",Abuse of Office,synthetic-abuse-of-office-03,Kiswahili,Kisumu\nA local official allegedly directed public workers to campaign during their paid government hours.,Abuse of Office,synthetic-abuse-of-office-04,English,Nakuru\nThe report states that a local official allegedly directed public workers to campaign during their paid government hours.,Abuse of Office,synthetic-abuse-of-office-04,English,Nakuru\nAfisa wa eneo anadaiwa kuwaelekeza wafanyakazi wa umma kufanya kampeni wakati wa saa za kazi za serikali.,Abuse of Office,synthetic-abuse-of-office-04,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, afisa wa eneo anadaiwa kuwaelekeza wafanyakazi wa umma kufanya kampeni wakati wa saa za kazi za serikali.",Abuse of Office,synthetic-abuse-of-office-04,Kiswahili,Nakuru\nA department head allegedly used an official vehicle and staff for a personal household task.,Abuse of Office,synthetic-abuse-of-office-05,English,Turkana\nThe report states that a department head allegedly used an official vehicle and staff for a personal household task.,Abuse of Office,synthetic-abuse-of-office-05,English,Turkana\nMkuu wa idara anadaiwa kutumia gari rasmi na wafanyakazi kwa kazi ya nyumbani binafsi.,Abuse of Office,synthetic-abuse-of-office-05,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, mkuu wa idara anadaiwa kutumia gari rasmi na wafanyakazi kwa kazi ya nyumbani binafsi.",Abuse of Office,synthetic-abuse-of-office-05,Kiswahili,Turkana\nA public officer allegedly blocked access to a record without applying the ordinary access rules.,Abuse of Office,synthetic-abuse-of-office-06,English,Garissa\nThe report states that a public officer allegedly blocked access to a record without applying the ordinary access rules.,Abuse of Office,synthetic-abuse-of-office-06,English,Garissa\nAfisa wa umma anadaiwa kuzuia upatikanaji wa rekodi bila kutumia kanuni za kawaida za kutoa taarifa.,Abuse of Office,synthetic-abuse-of-office-06,Kiswahili,Garissa\n"Kwa mujibu wa taarifa, afisa wa umma anadaiwa kuzuia upatikanaji wa rekodi bila kutumia kanuni za kawaida za kutoa taarifa.",Abuse of Office,synthetic-abuse-of-office-06,Kiswahili,Garissa\nA supervisor allegedly ordered a team to alter a routine inspection schedule to inconvenience one resident.,Abuse of Office,synthetic-abuse-of-office-07,English,Kilifi\nThe report states that a supervisor allegedly ordered a team to alter a routine inspection schedule to inconvenience one resident.,Abuse of Office,synthetic-abuse-of-office-07,English,Kilifi\nMsimamizi anadaiwa kuagiza timu kubadilisha ratiba ya ukaguzi ili kumsumbua mkazi mmoja.,Abuse of Office,synthetic-abuse-of-office-07,Kiswahili,Kilifi\n"Kwa mujibu wa taarifa, msimamizi anadaiwa kuagiza timu kubadilisha ratiba ya ukaguzi ili kumsumbua mkazi mmoja.",Abuse of Office,synthetic-abuse-of-office-07,Kiswahili,Kilifi\nA public employee allegedly used an official stamp to approve a personal transaction outside their duties.,Abuse of Office,synthetic-abuse-of-office-08,English,Bungoma\nThe report states that a public employee allegedly used an official stamp to approve a personal transaction outside their duties.,Abuse of Office,synthetic-abuse-of-office-08,English,Bungoma\nMfanyakazi wa umma anadaiwa kutumia muhuri rasmi kuidhinisha shughuli binafsi nje ya majukumu yake.,Abuse of Office,synthetic-abuse-of-office-08,Kiswahili,Bungoma\n"Kwa mujibu wa taarifa, mfanyakazi wa umma anadaiwa kutumia muhuri rasmi kuidhinisha shughuli binafsi nje ya majukumu yake.",Abuse of Office,synthetic-abuse-of-office-08,Kiswahili,Bungoma\nA manager reportedly directed public staff to perform work at a private property during office hours.,Abuse of Office,synthetic-abuse-of-office-09,English,Meru\nThe report states that a manager reportedly directed public staff to perform work at a private property during office hours.,Abuse of Office,synthetic-abuse-of-office-09,English,Meru\nMsimamizi anaripotiwa kuwaelekeza wafanyakazi wa umma kufanya kazi kwenye mali binafsi wakati wa saa za ofisi.,Abuse of Office,synthetic-abuse-of-office-09,Kiswahili,Meru\n"Kwa mujibu wa taarifa, msimamizi anaripotiwa kuwaelekeza wafanyakazi wa umma kufanya kazi kwenye mali binafsi wakati wa saa za ofisi.",Abuse of Office,synthetic-abuse-of-office-09,Kiswahili,Meru\nAn official allegedly used their position to cancel a citizen\'s valid application without a recorded reason.,Abuse of Office,synthetic-abuse-of-office-10,English,Kitui\nThe report states that an official allegedly used their position to cancel a citizen\'s valid application without a recorded reason.,Abuse of Office,synthetic-abuse-of-office-10,English,Kitui\nAfisa anadaiwa kutumia nafasi yake kufuta ombi halali la mwananchi bila sababu iliyoandikwa.,Abuse of Office,synthetic-abuse-of-office-10,Kiswahili,Kitui\n"Kwa mujibu wa taarifa, afisa anadaiwa kutumia nafasi yake kufuta ombi halali la mwananchi bila sababu iliyoandikwa.",Abuse of Office,synthetic-abuse-of-office-10,Kiswahili,Kitui\nA public supervisor allegedly instructed staff to ignore the normal queue for a personal acquaintance.,Abuse of Office,synthetic-abuse-of-office-11,English,Nairobi\nThe report states that a public supervisor allegedly instructed staff to ignore the normal queue for a personal acquaintance.,Abuse of Office,synthetic-abuse-of-office-11,English,Nairobi\nMsimamizi wa umma anadaiwa kuwaagiza wafanyakazi kupuuza foleni ya kawaida kwa ajili ya mtu anayemfahamu.,Abuse of Office,synthetic-abuse-of-office-11,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, msimamizi wa umma anadaiwa kuwaagiza wafanyakazi kupuuza foleni ya kawaida kwa ajili ya mtu anayemfahamu.",Abuse of Office,synthetic-abuse-of-office-11,Kiswahili,Nairobi\nA public officer reportedly used an office account to promote a personal business.,Abuse of Office,synthetic-abuse-of-office-12,English,Mombasa\nThe report states that a public officer reportedly used an office account to promote a personal business.,Abuse of Office,synthetic-abuse-of-office-12,English,Mombasa\nAfisa wa umma anaripotiwa kutumia akaunti ya ofisi kutangaza biashara yake binafsi.,Abuse of Office,synthetic-abuse-of-office-12,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, afisa wa umma anaripotiwa kutumia akaunti ya ofisi kutangaza biashara yake binafsi.",Abuse of Office,synthetic-abuse-of-office-12,Kiswahili,Mombasa\nA manager allegedly ordered the removal of a public notice to conceal a lawful community meeting.,Abuse of Office,synthetic-abuse-of-office-13,English,Kisumu\nThe report states that a manager allegedly ordered the removal of a public notice to conceal a lawful community meeting.,Abuse of Office,synthetic-abuse-of-office-13,English,Kisumu\nMsimamizi anadaiwa kuagiza kuondolewa kwa tangazo la umma ili kuficha mkutano halali wa jamii.,Abuse of Office,synthetic-abuse-of-office-13,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, msimamizi anadaiwa kuagiza kuondolewa kwa tangazo la umma ili kuficha mkutano halali wa jamii.",Abuse of Office,synthetic-abuse-of-office-13,Kiswahili,Kisumu\nAn employee allegedly used official authority to deny a routine inspection to a business without cause.,Abuse of Office,synthetic-abuse-of-office-14,English,Nakuru\nThe report states that an employee allegedly used official authority to deny a routine inspection to a business without cause.,Abuse of Office,synthetic-abuse-of-office-14,English,Nakuru\nMfanyakazi anadaiwa kutumia mamlaka rasmi kunyima biashara ukaguzi wa kawaida bila sababu.,Abuse of Office,synthetic-abuse-of-office-14,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, mfanyakazi anadaiwa kutumia mamlaka rasmi kunyima biashara ukaguzi wa kawaida bila sababu.",Abuse of Office,synthetic-abuse-of-office-14,Kiswahili,Nakuru\nA department officer reportedly used public staff and transport to carry personal household goods.,Abuse of Office,synthetic-abuse-of-office-15,English,Turkana\nThe report states that a department officer reportedly used public staff and transport to carry personal household goods.,Abuse of Office,synthetic-abuse-of-office-15,English,Turkana\nAfisa wa idara anaripotiwa kutumia wafanyakazi na usafiri wa umma kubeba bidhaa za nyumbani binafsi.,Abuse of Office,synthetic-abuse-of-office-15,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, afisa wa idara anaripotiwa kutumia wafanyakazi na usafiri wa umma kubeba bidhaa za nyumbani binafsi.",Abuse of Office,synthetic-abuse-of-office-15,Kiswahili,Turkana\nA qualified applicant alleges that a public job was given to the appointing manager\'s relative without a fair competition.,Nepotism/Favoritism,synthetic-nepotism-favoritism-01,English,Nairobi\nThe report states that a qualified applicant alleges that a public job was given to the appointing manager\'s relative without a fair competition.,Nepotism/Favoritism,synthetic-nepotism-favoritism-01,English,Nairobi\nMwombaji mwenye sifa anadai kuwa kazi ya umma alipewa ndugu wa msimamizi aliyeteua bila ushindani wa haki.,Nepotism/Favoritism,synthetic-nepotism-favoritism-01,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, mwombaji mwenye sifa anadai kuwa kazi ya umma alipewa ndugu wa msimamizi aliyeteua bila ushindani wa haki.",Nepotism/Favoritism,synthetic-nepotism-favoritism-01,Kiswahili,Nairobi\nA school committee says a supply contract was awarded to a company owned by an official\'s close family member.,Nepotism/Favoritism,synthetic-nepotism-favoritism-02,English,Mombasa\nThe report states that a school committee says a supply contract was awarded to a company owned by an official\'s close family member.,Nepotism/Favoritism,synthetic-nepotism-favoritism-02,English,Mombasa\nKamati ya shule inasema mkataba wa vifaa ulitolewa kwa kampuni inayomilikiwa na mwanafamilia wa karibu wa afisa.,Nepotism/Favoritism,synthetic-nepotism-favoritism-02,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, kamati ya shule inasema mkataba wa vifaa ulitolewa kwa kampuni inayomilikiwa na mwanafamilia wa karibu wa afisa.",Nepotism/Favoritism,synthetic-nepotism-favoritism-02,Kiswahili,Mombasa\nResidents allege that a public scholarship was reserved for relatives of committee members despite published criteria.,Nepotism/Favoritism,synthetic-nepotism-favoritism-03,English,Kisumu\nThe report states that residents allege that a public scholarship was reserved for relatives of committee members despite published criteria.,Nepotism/Favoritism,synthetic-nepotism-favoritism-03,English,Kisumu\nWakazi wanadai kuwa ufadhili wa masomo wa umma ulihifadhiwa kwa ndugu wa wajumbe licha ya vigezo vilivyochapishwa.,Nepotism/Favoritism,synthetic-nepotism-favoritism-03,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, wakazi wanadai kuwa ufadhili wa masomo wa umma ulihifadhiwa kwa ndugu wa wajumbe licha ya vigezo vilivyochapishwa.",Nepotism/Favoritism,synthetic-nepotism-favoritism-03,Kiswahili,Kisumu\nA recruitment shortlist allegedly included an officer\'s cousin who did not meet the stated minimum qualifications.,Nepotism/Favoritism,synthetic-nepotism-favoritism-04,English,Nakuru\nThe report states that a recruitment shortlist allegedly included an officer\'s cousin who did not meet the stated minimum qualifications.,Nepotism/Favoritism,synthetic-nepotism-favoritism-04,English,Nakuru\nOrodha fupi ya ajira inadaiwa kujumuisha binamu wa afisa ambaye hakutimiza sifa za chini zilizotajwa.,Nepotism/Favoritism,synthetic-nepotism-favoritism-04,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, orodha fupi ya ajira inadaiwa kujumuisha binamu wa afisa ambaye hakutimiza sifa za chini zilizotajwa.",Nepotism/Favoritism,synthetic-nepotism-favoritism-04,Kiswahili,Nakuru\nSeveral traders say market spaces were allocated to friends of the allocation panel ahead of eligible applicants.,Nepotism/Favoritism,synthetic-nepotism-favoritism-05,English,Turkana\nThe report states that several traders say market spaces were allocated to friends of the allocation panel ahead of eligible applicants.,Nepotism/Favoritism,synthetic-nepotism-favoritism-05,English,Turkana\nWafanyabiashara kadhaa wanasema nafasi za soko zilitolewa kwa marafiki wa jopo kabla ya waombaji waliostahili.,Nepotism/Favoritism,synthetic-nepotism-favoritism-05,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, wafanyabiashara kadhaa wanasema nafasi za soko zilitolewa kwa marafiki wa jopo kabla ya waombaji waliostahili.",Nepotism/Favoritism,synthetic-nepotism-favoritism-05,Kiswahili,Turkana\nA public committee allegedly selected a relative of one member for a grant while excluding equally eligible groups.,Nepotism/Favoritism,synthetic-nepotism-favoritism-06,English,Garissa\nThe report states that a public committee allegedly selected a relative of one member for a grant while excluding equally eligible groups.,Nepotism/Favoritism,synthetic-nepotism-favoritism-06,English,Garissa\nKamati ya umma inadaiwa kuchagua ndugu wa mjumbe mmoja kwa ruzuku huku ikiondoa vikundi vingine vilivyostahili.,Nepotism/Favoritism,synthetic-nepotism-favoritism-06,Kiswahili,Garissa\n"Kwa mujibu wa taarifa, kamati ya umma inadaiwa kuchagua ndugu wa mjumbe mmoja kwa ruzuku huku ikiondoa vikundi vingine vilivyostahili.",Nepotism/Favoritism,synthetic-nepotism-favoritism-06,Kiswahili,Garissa\nAn applicant alleges that a vacant public post was filled by an official\'s family member without advertising the role.,Nepotism/Favoritism,synthetic-nepotism-favoritism-07,English,Kilifi\nThe report states that an applicant alleges that a vacant public post was filled by an official\'s family member without advertising the role.,Nepotism/Favoritism,synthetic-nepotism-favoritism-07,English,Kilifi\nMwombaji anadai kuwa nafasi ya umma ilijazwa na mwanafamilia wa afisa bila kutangaza nafasi hiyo.,Nepotism/Favoritism,synthetic-nepotism-favoritism-07,Kiswahili,Kilifi\n"Kwa mujibu wa taarifa, mwombaji anadai kuwa nafasi ya umma ilijazwa na mwanafamilia wa afisa bila kutangaza nafasi hiyo.",Nepotism/Favoritism,synthetic-nepotism-favoritism-07,Kiswahili,Kilifi\nA community group says food distribution lists favored the relatives of local selection officials.,Nepotism/Favoritism,synthetic-nepotism-favoritism-08,English,Bungoma\nThe report states that a community group says food distribution lists favored the relatives of local selection officials.,Nepotism/Favoritism,synthetic-nepotism-favoritism-08,English,Bungoma\nKikundi cha jamii kinasema orodha za ugawaji wa chakula ziliwapendelea ndugu wa maafisa wa uteuzi wa eneo.,Nepotism/Favoritism,synthetic-nepotism-favoritism-08,Kiswahili,Bungoma\n"Kwa mujibu wa taarifa, kikundi cha jamii kinasema orodha za ugawaji wa chakula ziliwapendelea ndugu wa maafisa wa uteuzi wa eneo.",Nepotism/Favoritism,synthetic-nepotism-favoritism-08,Kiswahili,Bungoma\nA public board allegedly gave repeated small contracts to a close associate while qualified applicants were overlooked.,Nepotism/Favoritism,synthetic-nepotism-favoritism-09,English,Meru\nThe report states that a public board allegedly gave repeated small contracts to a close associate while qualified applicants were overlooked.,Nepotism/Favoritism,synthetic-nepotism-favoritism-09,English,Meru\nBodi ya umma inadaiwa kumpa mikataba midogo mara kwa mara mshirika wa karibu huku waombaji wenye sifa wakipuuzwa.,Nepotism/Favoritism,synthetic-nepotism-favoritism-09,Kiswahili,Meru\n"Kwa mujibu wa taarifa, bodi ya umma inadaiwa kumpa mikataba midogo mara kwa mara mshirika wa karibu huku waombaji wenye sifa wakipuuzwa.",Nepotism/Favoritism,synthetic-nepotism-favoritism-09,Kiswahili,Meru\nA resident reports that a training place was reserved for a committee member\'s relative outside the open selection.,Nepotism/Favoritism,synthetic-nepotism-favoritism-10,English,Kitui\nThe report states that a resident reports that a training place was reserved for a committee member\'s relative outside the open selection.,Nepotism/Favoritism,synthetic-nepotism-favoritism-10,English,Kitui\nMkazi anaripoti kuwa nafasi ya mafunzo ilihifadhiwa kwa ndugu wa mjumbe wa kamati nje ya uteuzi wa wazi.,Nepotism/Favoritism,synthetic-nepotism-favoritism-10,Kiswahili,Kitui\n"Kwa mujibu wa taarifa, mkazi anaripoti kuwa nafasi ya mafunzo ilihifadhiwa kwa ndugu wa mjumbe wa kamati nje ya uteuzi wa wazi.",Nepotism/Favoritism,synthetic-nepotism-favoritism-10,Kiswahili,Kitui\nAn employment panel allegedly advanced a family member of the chairperson despite a lower assessment score.,Nepotism/Favoritism,synthetic-nepotism-favoritism-11,English,Nairobi\nThe report states that an employment panel allegedly advanced a family member of the chairperson despite a lower assessment score.,Nepotism/Favoritism,synthetic-nepotism-favoritism-11,English,Nairobi\nJopo la ajira linadaiwa kumpandisha mwanafamilia wa mwenyekiti licha ya alama ya tathmini kuwa chini.,Nepotism/Favoritism,synthetic-nepotism-favoritism-11,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, jopo la ajira linadaiwa kumpandisha mwanafamilia wa mwenyekiti licha ya alama ya tathmini kuwa chini.",Nepotism/Favoritism,synthetic-nepotism-favoritism-11,Kiswahili,Nairobi\nA public project allegedly hired a supplier owned by an official\'s spouse without an open comparison of offers.,Nepotism/Favoritism,synthetic-nepotism-favoritism-12,English,Mombasa\nThe report states that a public project allegedly hired a supplier owned by an official\'s spouse without an open comparison of offers.,Nepotism/Favoritism,synthetic-nepotism-favoritism-12,English,Mombasa\nMradi wa umma unadaiwa kuajiri msambazaji anayemilikiwa na mwenzi wa afisa bila kulinganisha ofa kwa uwazi.,Nepotism/Favoritism,synthetic-nepotism-favoritism-12,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, mradi wa umma unadaiwa kuajiri msambazaji anayemilikiwa na mwenzi wa afisa bila kulinganisha ofa kwa uwazi.",Nepotism/Favoritism,synthetic-nepotism-favoritism-12,Kiswahili,Mombasa\nResidents allege that temporary public jobs were distributed mainly among relatives of the local committee.,Nepotism/Favoritism,synthetic-nepotism-favoritism-13,English,Kisumu\nThe report states that residents allege that temporary public jobs were distributed mainly among relatives of the local committee.,Nepotism/Favoritism,synthetic-nepotism-favoritism-13,English,Kisumu\nWakazi wanadai kuwa kazi za muda za umma ziligawiwa hasa kwa ndugu wa kamati ya eneo.,Nepotism/Favoritism,synthetic-nepotism-favoritism-13,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, wakazi wanadai kuwa kazi za muda za umma ziligawiwa hasa kwa ndugu wa kamati ya eneo.",Nepotism/Favoritism,synthetic-nepotism-favoritism-13,Kiswahili,Kisumu\nA grant panel reportedly favored an official\'s sibling over applicants who met the same published requirements.,Nepotism/Favoritism,synthetic-nepotism-favoritism-14,English,Nakuru\nThe report states that a grant panel reportedly favored an official\'s sibling over applicants who met the same published requirements.,Nepotism/Favoritism,synthetic-nepotism-favoritism-14,English,Nakuru\nJopo la ruzuku linaripotiwa kumpendelea ndugu wa afisa kuliko waombaji waliotimiza masharti yale yale.,Nepotism/Favoritism,synthetic-nepotism-favoritism-14,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, jopo la ruzuku linaripotiwa kumpendelea ndugu wa afisa kuliko waombaji waliotimiza masharti yale yale.",Nepotism/Favoritism,synthetic-nepotism-favoritism-14,Kiswahili,Nakuru\nA qualified candidate says a public appointment went to a close friend of the selection officer without a fair process.,Nepotism/Favoritism,synthetic-nepotism-favoritism-15,English,Turkana\nThe report states that a qualified candidate says a public appointment went to a close friend of the selection officer without a fair process.,Nepotism/Favoritism,synthetic-nepotism-favoritism-15,English,Turkana\nMgombea mwenye sifa anasema uteuzi wa umma ulienda kwa rafiki wa karibu wa afisa wa uteuzi bila utaratibu wa haki.,Nepotism/Favoritism,synthetic-nepotism-favoritism-15,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, mgombea mwenye sifa anasema uteuzi wa umma ulienda kwa rafiki wa karibu wa afisa wa uteuzi bila utaratibu wa haki.",Nepotism/Favoritism,synthetic-nepotism-favoritism-15,Kiswahili,Turkana\nA trader says an official threatened to close a lawful stall unless the trader paid an unofficial amount.,Extortion,synthetic-extortion-01,English,Nairobi\nThe report states that a trader says an official threatened to close a lawful stall unless the trader paid an unofficial amount.,Extortion,synthetic-extortion-01,English,Nairobi\nMfanyabiashara anasema afisa alitishia kufunga kibanda halali isipokuwa alipe kiasi kisicho rasmi.,Extortion,synthetic-extortion-01,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, mfanyabiashara anasema afisa alitishia kufunga kibanda halali isipokuwa alipe kiasi kisicho rasmi.",Extortion,synthetic-extortion-01,Kiswahili,Nairobi\nA resident alleges that a clerk threatened to destroy an application unless a personal payment was made.,Extortion,synthetic-extortion-02,English,Mombasa\nThe report states that a resident alleges that a clerk threatened to destroy an application unless a personal payment was made.,Extortion,synthetic-extortion-02,English,Mombasa\nMkazi anadai kuwa karani alitishia kuharibu ombi isipokuwa malipo binafsi yafanywe.,Extortion,synthetic-extortion-02,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, mkazi anadai kuwa karani alitishia kuharibu ombi isipokuwa malipo binafsi yafanywe.",Extortion,synthetic-extortion-02,Kiswahili,Mombasa\nA public inspector allegedly demanded money and threatened to seize equipment if the owner refused.,Extortion,synthetic-extortion-03,English,Kisumu\nThe report states that a public inspector allegedly demanded money and threatened to seize equipment if the owner refused.,Extortion,synthetic-extortion-03,English,Kisumu\nMkaguzi wa umma anadaiwa kudai pesa na kutishia kukamata vifaa ikiwa mmiliki atakataa.,Extortion,synthetic-extortion-03,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, mkaguzi wa umma anadaiwa kudai pesa na kutishia kukamata vifaa ikiwa mmiliki atakataa.",Extortion,synthetic-extortion-03,Kiswahili,Kisumu\nA contractor says a supervisor threatened to withhold an approved certificate unless an unofficial fee was paid.,Extortion,synthetic-extortion-04,English,Nakuru\nThe report states that a contractor says a supervisor threatened to withhold an approved certificate unless an unofficial fee was paid.,Extortion,synthetic-extortion-04,English,Nakuru\nMkandarasi anasema msimamizi alitishia kuzuia cheti kilichoidhinishwa isipokuwa ada isiyo rasmi ilipwe.,Extortion,synthetic-extortion-04,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, mkandarasi anasema msimamizi alitishia kuzuia cheti kilichoidhinishwa isipokuwa ada isiyo rasmi ilipwe.",Extortion,synthetic-extortion-04,Kiswahili,Nakuru\nA service user reports being threatened with removal from a public queue for refusing a demanded payment.,Extortion,synthetic-extortion-05,English,Turkana\nThe report states that a service user reports being threatened with removal from a public queue for refusing a demanded payment.,Extortion,synthetic-extortion-05,English,Turkana\nMtumiaji wa huduma anaripoti kutishiwa kuondolewa kwenye foleni ya umma kwa kukataa malipo yaliyodaiwa.,Extortion,synthetic-extortion-05,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, mtumiaji wa huduma anaripoti kutishiwa kuondolewa kwenye foleni ya umma kwa kukataa malipo yaliyodaiwa.",Extortion,synthetic-extortion-05,Kiswahili,Turkana\nAn officer allegedly said a resident would face a false penalty unless money was handed over privately.,Extortion,synthetic-extortion-06,English,Garissa\nThe report states that an officer allegedly said a resident would face a false penalty unless money was handed over privately.,Extortion,synthetic-extortion-06,English,Garissa\nAfisa anadaiwa kusema mkazi atatozwa adhabu ya uongo isipokuwa pesa zikabidhiwe kwa siri.,Extortion,synthetic-extortion-06,Kiswahili,Garissa\n"Kwa mujibu wa taarifa, afisa anadaiwa kusema mkazi atatozwa adhabu ya uongo isipokuwa pesa zikabidhiwe kwa siri.",Extortion,synthetic-extortion-06,Kiswahili,Garissa\nA business owner says a public employee threatened to delay renewal unless a demanded personal fee was paid.,Extortion,synthetic-extortion-07,English,Kilifi\nThe report states that a business owner says a public employee threatened to delay renewal unless a demanded personal fee was paid.,Extortion,synthetic-extortion-07,English,Kilifi\nMmiliki wa biashara anasema mfanyakazi wa umma alitishia kuchelewesha upya wa leseni isipokuwa ada binafsi ilipwe.,Extortion,synthetic-extortion-07,Kiswahili,Kilifi\n"Kwa mujibu wa taarifa, mmiliki wa biashara anasema mfanyakazi wa umma alitishia kuchelewesha upya wa leseni isipokuwa ada binafsi ilipwe.",Extortion,synthetic-extortion-07,Kiswahili,Kilifi\nA resident alleges that an official threatened to report them unlawfully unless they paid for a routine service.,Extortion,synthetic-extortion-08,English,Bungoma\nThe report states that a resident alleges that an official threatened to report them unlawfully unless they paid for a routine service.,Extortion,synthetic-extortion-08,English,Bungoma\nMkazi anadai kuwa afisa alitishia kumripoti kinyume cha sheria isipokuwa alipe kwa huduma ya kawaida.,Extortion,synthetic-extortion-08,Kiswahili,Bungoma\n"Kwa mujibu wa taarifa, mkazi anadai kuwa afisa alitishia kumripoti kinyume cha sheria isipokuwa alipe kwa huduma ya kawaida.",Extortion,synthetic-extortion-08,Kiswahili,Bungoma\nA vendor says a public officer threatened to confiscate goods unless the vendor paid money outside the official process.,Extortion,synthetic-extortion-09,English,Meru\nThe report states that a vendor says a public officer threatened to confiscate goods unless the vendor paid money outside the official process.,Extortion,synthetic-extortion-09,English,Meru\nMuuzaji anasema afisa wa umma alitishia kuchukua bidhaa isipokuwa alipe pesa nje ya utaratibu rasmi.,Extortion,synthetic-extortion-09,Kiswahili,Meru\n"Kwa mujibu wa taarifa, muuzaji anasema afisa wa umma alitishia kuchukua bidhaa isipokuwa alipe pesa nje ya utaratibu rasmi.",Extortion,synthetic-extortion-09,Kiswahili,Meru\nA worker alleges that a supervisor threatened to block a lawful payment unless part of it was returned privately.,Extortion,synthetic-extortion-10,English,Kitui\nThe report states that a worker alleges that a supervisor threatened to block a lawful payment unless part of it was returned privately.,Extortion,synthetic-extortion-10,English,Kitui\nMfanyakazi anadai kuwa msimamizi alitishia kuzuia malipo halali isipokuwa sehemu irudishwe kwa siri.,Extortion,synthetic-extortion-10,Kiswahili,Kitui\n"Kwa mujibu wa taarifa, mfanyakazi anadai kuwa msimamizi alitishia kuzuia malipo halali isipokuwa sehemu irudishwe kwa siri.",Extortion,synthetic-extortion-10,Kiswahili,Kitui\nA resident says an officer threatened arrest unless an unofficial payment was made.,Extortion,synthetic-extortion-11,English,Nairobi\nThe report states that a resident says an officer threatened arrest unless an unofficial payment was made.,Extortion,synthetic-extortion-11,English,Nairobi\nMkazi anasema afisa alitishia kumkamata isipokuwa malipo yasiyo rasmi yafanywe.,Extortion,synthetic-extortion-11,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, mkazi anasema afisa alitishia kumkamata isipokuwa malipo yasiyo rasmi yafanywe.",Extortion,synthetic-extortion-11,Kiswahili,Nairobi\nA supplier alleges that a public employee threatened to blacklist the business unless a personal demand was met.,Extortion,synthetic-extortion-12,English,Mombasa\nThe report states that a supplier alleges that a public employee threatened to blacklist the business unless a personal demand was met.,Extortion,synthetic-extortion-12,English,Mombasa\nMsambazaji anadai kuwa mfanyakazi wa umma alitishia kuorodhesha biashara kama hairuhusiwi isipokuwa dai binafsi litimizwe.,Extortion,synthetic-extortion-12,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, msambazaji anadai kuwa mfanyakazi wa umma alitishia kuorodhesha biashara kama hairuhusiwi isipokuwa dai binafsi litimizwe.",Extortion,synthetic-extortion-12,Kiswahili,Mombasa\nA market trader says an inspector threatened to remove their stall unless a private payment was handed over.,Extortion,synthetic-extortion-13,English,Kisumu\nThe report states that a market trader says an inspector threatened to remove their stall unless a private payment was handed over.,Extortion,synthetic-extortion-13,English,Kisumu\nMfanyabiashara wa soko anasema mkaguzi alitishia kuondoa kibanda isipokuwa malipo ya faragha yatolewe.,Extortion,synthetic-extortion-13,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, mfanyabiashara wa soko anasema mkaguzi alitishia kuondoa kibanda isipokuwa malipo ya faragha yatolewe.",Extortion,synthetic-extortion-13,Kiswahili,Kisumu\nAn applicant reports that an official threatened to discard valid papers unless the applicant paid cash.,Extortion,synthetic-extortion-14,English,Nakuru\nThe report states that an applicant reports that an official threatened to discard valid papers unless the applicant paid cash.,Extortion,synthetic-extortion-14,English,Nakuru\nMwombaji anaripoti kuwa afisa alitishia kutupa nyaraka halali isipokuwa mwombaji alipe pesa taslimu.,Extortion,synthetic-extortion-14,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, mwombaji anaripoti kuwa afisa alitishia kutupa nyaraka halali isipokuwa mwombaji alipe pesa taslimu.",Extortion,synthetic-extortion-14,Kiswahili,Nakuru\nA public worker allegedly threatened to cancel a service request unless the user provided a personal benefit.,Extortion,synthetic-extortion-15,English,Turkana\nThe report states that a public worker allegedly threatened to cancel a service request unless the user provided a personal benefit.,Extortion,synthetic-extortion-15,English,Turkana\nMfanyakazi wa umma anadaiwa kutishia kufuta ombi la huduma isipokuwa mtumiaji atoe manufaa binafsi.,Extortion,synthetic-extortion-15,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, mfanyakazi wa umma anadaiwa kutishia kufuta ombi la huduma isipokuwa mtumiaji atoe manufaa binafsi.",Extortion,synthetic-extortion-15,Kiswahili,Turkana\nA resident alleges that an official receipt was altered to show a different payment amount.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-01,English,Nairobi\nThe report states that a resident alleges that an official receipt was altered to show a different payment amount.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-01,English,Nairobi\nMkazi anadai kuwa risiti rasmi ilibadilishwa ili kuonyesha kiasi tofauti cha malipo.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-01,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, mkazi anadai kuwa risiti rasmi ilibadilishwa ili kuonyesha kiasi tofauti cha malipo.",Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-01,Kiswahili,Nairobi\nA public project file allegedly contains a completion certificate for work that was not performed.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-02,English,Mombasa\nThe report states that a public project file allegedly contains a completion certificate for work that was not performed.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-02,English,Mombasa\nFaili ya mradi wa umma inadaiwa kuwa na cheti cha kukamilika kwa kazi ambayo haikufanywa.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-02,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, faili ya mradi wa umma inadaiwa kuwa na cheti cha kukamilika kwa kazi ambayo haikufanywa.",Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-02,Kiswahili,Mombasa\nAn applicant says a signature on a public approval letter was copied without the signatory\'s permission.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-03,English,Kisumu\nThe report states that an applicant says a signature on a public approval letter was copied without the signatory\'s permission.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-03,English,Kisumu\nMwombaji anasema sahihi kwenye barua ya idhini ya umma ilinakiliwa bila ruhusa ya aliyeweka sahihi.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-03,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, mwombaji anasema sahihi kwenye barua ya idhini ya umma ilinakiliwa bila ruhusa ya aliyeweka sahihi.",Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-03,Kiswahili,Kisumu\nA supplier alleges that a delivery record was changed to claim goods were received when they were not.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-04,English,Nakuru\nThe report states that a supplier alleges that a delivery record was changed to claim goods were received when they were not.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-04,English,Nakuru\nMsambazaji anadai kuwa rekodi ya uwasilishaji ilibadilishwa kuonyesha bidhaa zilipokelewa ilhali hazikupokelewa.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-04,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, msambazaji anadai kuwa rekodi ya uwasilishaji ilibadilishwa kuonyesha bidhaa zilipokelewa ilhali hazikupokelewa.",Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-04,Kiswahili,Nakuru\nA public register reportedly lists a permit number that the issuing office says it never created.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-05,English,Turkana\nThe report states that a public register reportedly lists a permit number that the issuing office says it never created.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-05,English,Turkana\nDaftari la umma linaripotiwa kuonyesha nambari ya kibali ambayo ofisi inayotoa vibali inasema haikuunda.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-05,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, daftari la umma linaripotiwa kuonyesha nambari ya kibali ambayo ofisi inayotoa vibali inasema haikuunda.",Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-05,Kiswahili,Turkana\nA committee says a meeting minute was edited to record an approval that members did not give.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-06,English,Garissa\nThe report states that a committee says a meeting minute was edited to record an approval that members did not give.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-06,English,Garissa\nKamati inasema kumbukumbu za mkutano zilihaririwa kurekodi idhini ambayo wajumbe hawakutoa.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-06,Kiswahili,Garissa\n"Kwa mujibu wa taarifa, kamati inasema kumbukumbu za mkutano zilihaririwa kurekodi idhini ambayo wajumbe hawakutoa.",Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-06,Kiswahili,Garissa\nA claimant alleges that a forged letter was used to obtain a public benefit under a false identity.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-07,English,Kilifi\nThe report states that a claimant alleges that a forged letter was used to obtain a public benefit under a false identity.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-07,English,Kilifi\nMlipaji anadai kuwa barua ya kughushi ilitumika kupata manufaa ya umma kwa utambulisho wa uongo.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-07,Kiswahili,Kilifi\n"Kwa mujibu wa taarifa, mlipaji anadai kuwa barua ya kughushi ilitumika kupata manufaa ya umma kwa utambulisho wa uongo.",Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-07,Kiswahili,Kilifi\nAn audit review allegedly found duplicate signatures on separate public payment vouchers.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-08,English,Bungoma\nThe report states that an audit review allegedly found duplicate signatures on separate public payment vouchers.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-08,English,Bungoma\nUkaguzi unadaiwa kupata sahihi zilizorudiwa kwenye hati tofauti za malipo ya umma.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-08,Kiswahili,Bungoma\n"Kwa mujibu wa taarifa, ukaguzi unadaiwa kupata sahihi zilizorudiwa kwenye hati tofauti za malipo ya umma.",Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-08,Kiswahili,Bungoma\nA resident says a land-record entry was falsified to make an unapproved transfer appear official.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-09,English,Meru\nThe report states that a resident says a land-record entry was falsified to make an unapproved transfer appear official.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-09,English,Meru\nMkazi anasema rekodi ya ardhi ilighushiwa ili uhamisho usioidhinishwa uonekane rasmi.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-09,Kiswahili,Meru\n"Kwa mujibu wa taarifa, mkazi anasema rekodi ya ardhi ilighushiwa ili uhamisho usioidhinishwa uonekane rasmi.",Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-09,Kiswahili,Meru\nA public office allegedly issued a certificate bearing a fabricated inspection result.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-10,English,Kitui\nThe report states that a public office allegedly issued a certificate bearing a fabricated inspection result.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-10,English,Kitui\nOfisi ya umma inadaiwa kutoa cheti chenye matokeo ya ukaguzi yaliyobuniwa.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-10,Kiswahili,Kitui\n"Kwa mujibu wa taarifa, ofisi ya umma inadaiwa kutoa cheti chenye matokeo ya ukaguzi yaliyobuniwa.",Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-10,Kiswahili,Kitui\nA payment file reportedly includes a fake invoice submitted as evidence of a public purchase.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-11,English,Nairobi\nThe report states that a payment file reportedly includes a fake invoice submitted as evidence of a public purchase.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-11,English,Nairobi\nFaili ya malipo inaripotiwa kuwa na ankara bandia iliyowasilishwa kama ushahidi wa ununuzi wa umma.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-11,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, faili ya malipo inaripotiwa kuwa na ankara bandia iliyowasilishwa kama ushahidi wa ununuzi wa umma.",Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-11,Kiswahili,Nairobi\nA resident alleges that an official altered a date on a public notice after the response deadline had passed.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-12,English,Mombasa\nThe report states that a resident alleges that an official altered a date on a public notice after the response deadline had passed.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-12,English,Mombasa\nMkazi anadai kuwa afisa alibadilisha tarehe kwenye tangazo la umma baada ya muda wa kujibu kupita.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-12,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, mkazi anadai kuwa afisa alibadilisha tarehe kwenye tangazo la umma baada ya muda wa kujibu kupita.",Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-12,Kiswahili,Mombasa\nA public application reportedly carries an approval stamp that the issuing department says is not genuine.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-13,English,Kisumu\nThe report states that a public application reportedly carries an approval stamp that the issuing department says is not genuine.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-13,English,Kisumu\nOmbi la umma linaripotiwa kuwa na muhuri wa idhini ambao idara inayotoa inasema si halisi.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-13,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, ombi la umma linaripotiwa kuwa na muhuri wa idhini ambao idara inayotoa inasema si halisi.",Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-13,Kiswahili,Kisumu\nA committee alleges that a record was fabricated to show a public meeting had taken place.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-14,English,Nakuru\nThe report states that a committee alleges that a record was fabricated to show a public meeting had taken place.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-14,English,Nakuru\nKamati inadai kuwa rekodi ilibuniwa kuonyesha kuwa mkutano wa umma ulifanyika.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-14,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, kamati inadai kuwa rekodi ilibuniwa kuonyesha kuwa mkutano wa umma ulifanyika.",Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-14,Kiswahili,Nakuru\nAn applicant says an official document was changed to remove a required condition from the approved version.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-15,English,Turkana\nThe report states that an applicant says an official document was changed to remove a required condition from the approved version.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-15,English,Turkana\nMwombaji anasema hati rasmi ilibadilishwa kuondoa sharti lililohitajika kutoka toleo lililoidhinishwa.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-15,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, mwombaji anasema hati rasmi ilibadilishwa kuondoa sharti lililohitajika kutoka toleo lililoidhinishwa.",Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-15,Kiswahili,Turkana\nA committee member allegedly voted on a contract that would pay a company in which the member owns shares.,Conflict of Interest,synthetic-conflict-of-interest-01,English,Nairobi\nThe report states that a committee member allegedly voted on a contract that would pay a company in which the member owns shares.,Conflict of Interest,synthetic-conflict-of-interest-01,English,Nairobi\nMjumbe wa kamati anadaiwa kupiga kura kuhusu mkataba ambao ungelipa kampuni ambayo ana hisa.,Conflict of Interest,synthetic-conflict-of-interest-01,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, mjumbe wa kamati anadaiwa kupiga kura kuhusu mkataba ambao ungelipa kampuni ambayo ana hisa.",Conflict of Interest,synthetic-conflict-of-interest-01,Kiswahili,Nairobi\nA public officer reportedly helped select a supplier while privately holding a financial interest in that business.,Conflict of Interest,synthetic-conflict-of-interest-02,English,Mombasa\nThe report states that a public officer reportedly helped select a supplier while privately holding a financial interest in that business.,Conflict of Interest,synthetic-conflict-of-interest-02,English,Mombasa\nAfisa wa umma anaripotiwa kusaidia kuchagua msambazaji huku akiwa na maslahi ya kifedha ya siri katika biashara hiyo.,Conflict of Interest,synthetic-conflict-of-interest-02,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, afisa wa umma anaripotiwa kusaidia kuchagua msambazaji huku akiwa na maslahi ya kifedha ya siri katika biashara hiyo.",Conflict of Interest,synthetic-conflict-of-interest-02,Kiswahili,Mombasa\nA manager allegedly approved a public lease for land they personally own without disclosing the ownership.,Conflict of Interest,synthetic-conflict-of-interest-03,English,Kisumu\nThe report states that a manager allegedly approved a public lease for land they personally own without disclosing the ownership.,Conflict of Interest,synthetic-conflict-of-interest-03,English,Kisumu\nMsimamizi anadaiwa kuidhinisha upangishaji wa umma wa ardhi anayomiliki bila kufichua umiliki huo.,Conflict of Interest,synthetic-conflict-of-interest-03,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, msimamizi anadaiwa kuidhinisha upangishaji wa umma wa ardhi anayomiliki bila kufichua umiliki huo.",Conflict of Interest,synthetic-conflict-of-interest-03,Kiswahili,Kisumu\nA board member allegedly took part in evaluating an application from a business where they serve as a director.,Conflict of Interest,synthetic-conflict-of-interest-04,English,Nakuru\nThe report states that a board member allegedly took part in evaluating an application from a business where they serve as a director.,Conflict of Interest,synthetic-conflict-of-interest-04,English,Nakuru\nMjumbe wa bodi anadaiwa kushiriki kutathmini ombi la biashara ambako yeye ni mkurugenzi.,Conflict of Interest,synthetic-conflict-of-interest-04,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, mjumbe wa bodi anadaiwa kushiriki kutathmini ombi la biashara ambako yeye ni mkurugenzi.",Conflict of Interest,synthetic-conflict-of-interest-04,Kiswahili,Nakuru\nAn officer allegedly shaped a public decision that would financially benefit their undisclosed private partnership.,Conflict of Interest,synthetic-conflict-of-interest-05,English,Turkana\nThe report states that an officer allegedly shaped a public decision that would financially benefit their undisclosed private partnership.,Conflict of Interest,synthetic-conflict-of-interest-05,English,Turkana\nAfisa anadaiwa kuelekeza uamuzi wa umma ambao ungeleta faida ya kifedha kwa ubia wake binafsi ambao hakufichua.,Conflict of Interest,synthetic-conflict-of-interest-05,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, afisa anadaiwa kuelekeza uamuzi wa umma ambao ungeleta faida ya kifedha kwa ubia wake binafsi ambao hakufichua.",Conflict of Interest,synthetic-conflict-of-interest-05,Kiswahili,Turkana\nA public employee reportedly evaluated a grant for an organization in which the employee holds a paid position.,Conflict of Interest,synthetic-conflict-of-interest-06,English,Garissa\nThe report states that a public employee reportedly evaluated a grant for an organization in which the employee holds a paid position.,Conflict of Interest,synthetic-conflict-of-interest-06,English,Garissa\nMfanyakazi wa umma anaripotiwa kutathmini ruzuku ya shirika ambako ana nafasi ya kulipwa.,Conflict of Interest,synthetic-conflict-of-interest-06,Kiswahili,Garissa\n"Kwa mujibu wa taarifa, mfanyakazi wa umma anaripotiwa kutathmini ruzuku ya shirika ambako ana nafasi ya kulipwa.",Conflict of Interest,synthetic-conflict-of-interest-06,Kiswahili,Garissa\nA procurement panel member allegedly reviewed bids from a company in which their spouse has a financial stake.,Conflict of Interest,synthetic-conflict-of-interest-07,English,Kilifi\nThe report states that a procurement panel member allegedly reviewed bids from a company in which their spouse has a financial stake.,Conflict of Interest,synthetic-conflict-of-interest-07,English,Kilifi\nMjumbe wa jopo la ununuzi anadaiwa kutathmini zabuni za kampuni ambako mwenzi wake ana maslahi ya kifedha.,Conflict of Interest,synthetic-conflict-of-interest-07,Kiswahili,Kilifi\n"Kwa mujibu wa taarifa, mjumbe wa jopo la ununuzi anadaiwa kutathmini zabuni za kampuni ambako mwenzi wake ana maslahi ya kifedha.",Conflict of Interest,synthetic-conflict-of-interest-07,Kiswahili,Kilifi\nA manager allegedly approved use of public property by a business they privately co-own.,Conflict of Interest,synthetic-conflict-of-interest-08,English,Bungoma\nThe report states that a manager allegedly approved use of public property by a business they privately co-own.,Conflict of Interest,synthetic-conflict-of-interest-08,English,Bungoma\nMsimamizi anadaiwa kuidhinisha matumizi ya mali ya umma na biashara anayomiliki kwa pamoja kwa siri.,Conflict of Interest,synthetic-conflict-of-interest-08,Kiswahili,Bungoma\n"Kwa mujibu wa taarifa, msimamizi anadaiwa kuidhinisha matumizi ya mali ya umma na biashara anayomiliki kwa pamoja kwa siri.",Conflict of Interest,synthetic-conflict-of-interest-08,Kiswahili,Bungoma\nA public board member reportedly influenced a permit decision involving their own private investment.,Conflict of Interest,synthetic-conflict-of-interest-09,English,Meru\nThe report states that a public board member reportedly influenced a permit decision involving their own private investment.,Conflict of Interest,synthetic-conflict-of-interest-09,English,Meru\nMjumbe wa bodi ya umma anaripotiwa kushawishi uamuzi wa kibali unaohusu uwekezaji wake binafsi.,Conflict of Interest,synthetic-conflict-of-interest-09,Kiswahili,Meru\n"Kwa mujibu wa taarifa, mjumbe wa bodi ya umma anaripotiwa kushawishi uamuzi wa kibali unaohusu uwekezaji wake binafsi.",Conflict of Interest,synthetic-conflict-of-interest-09,Kiswahili,Meru\nAn officer allegedly participated in setting a public contract price while owning part of the bidding company.,Conflict of Interest,synthetic-conflict-of-interest-10,English,Kitui\nThe report states that an officer allegedly participated in setting a public contract price while owning part of the bidding company.,Conflict of Interest,synthetic-conflict-of-interest-10,English,Kitui\nAfisa anadaiwa kushiriki kupanga bei ya mkataba wa umma huku akimiliki sehemu ya kampuni iliyotoa zabuni.,Conflict of Interest,synthetic-conflict-of-interest-10,Kiswahili,Kitui\n"Kwa mujibu wa taarifa, afisa anadaiwa kushiriki kupanga bei ya mkataba wa umma huku akimiliki sehemu ya kampuni iliyotoa zabuni.",Conflict of Interest,synthetic-conflict-of-interest-10,Kiswahili,Kitui\nA public employee reportedly approved a business application that would increase the value of their private property.,Conflict of Interest,synthetic-conflict-of-interest-11,English,Nairobi\nThe report states that a public employee reportedly approved a business application that would increase the value of their private property.,Conflict of Interest,synthetic-conflict-of-interest-11,English,Nairobi\nMfanyakazi wa umma anaripotiwa kuidhinisha ombi la biashara ambalo lingeongeza thamani ya mali yake binafsi.,Conflict of Interest,synthetic-conflict-of-interest-11,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, mfanyakazi wa umma anaripotiwa kuidhinisha ombi la biashara ambalo lingeongeza thamani ya mali yake binafsi.",Conflict of Interest,synthetic-conflict-of-interest-11,Kiswahili,Nairobi\nA committee member allegedly selected a consultant whose firm pays the member for separate private work.,Conflict of Interest,synthetic-conflict-of-interest-12,English,Mombasa\nThe report states that a committee member allegedly selected a consultant whose firm pays the member for separate private work.,Conflict of Interest,synthetic-conflict-of-interest-12,English,Mombasa\nMjumbe wa kamati anadaiwa kuchagua mshauri ambaye kampuni yake humlipa kwa kazi tofauti ya binafsi.,Conflict of Interest,synthetic-conflict-of-interest-12,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, mjumbe wa kamati anadaiwa kuchagua mshauri ambaye kampuni yake humlipa kwa kazi tofauti ya binafsi.",Conflict of Interest,synthetic-conflict-of-interest-12,Kiswahili,Mombasa\nAn officer reportedly took part in a public allocation decision affecting a company they privately advise for a fee.,Conflict of Interest,synthetic-conflict-of-interest-13,English,Kisumu\nThe report states that an officer reportedly took part in a public allocation decision affecting a company they privately advise for a fee.,Conflict of Interest,synthetic-conflict-of-interest-13,English,Kisumu\nAfisa anaripotiwa kushiriki uamuzi wa mgao wa umma unaoathiri kampuni anayoshauri kwa ada binafsi.,Conflict of Interest,synthetic-conflict-of-interest-13,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, afisa anaripotiwa kushiriki uamuzi wa mgao wa umma unaoathiri kampuni anayoshauri kwa ada binafsi.",Conflict of Interest,synthetic-conflict-of-interest-13,Kiswahili,Kisumu\nA board member allegedly voted on selling public equipment to a business in which they have a financial interest.,Conflict of Interest,synthetic-conflict-of-interest-14,English,Nakuru\nThe report states that a board member allegedly voted on selling public equipment to a business in which they have a financial interest.,Conflict of Interest,synthetic-conflict-of-interest-14,English,Nakuru\nMjumbe wa bodi anadaiwa kupiga kura kuhusu kuuza vifaa vya umma kwa biashara ambako ana maslahi ya kifedha.,Conflict of Interest,synthetic-conflict-of-interest-14,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, mjumbe wa bodi anadaiwa kupiga kura kuhusu kuuza vifaa vya umma kwa biashara ambako ana maslahi ya kifedha.",Conflict of Interest,synthetic-conflict-of-interest-14,Kiswahili,Nakuru\nA public manager allegedly influenced a land-use decision that would benefit an undisclosed personal investment.,Conflict of Interest,synthetic-conflict-of-interest-15,English,Turkana\nThe report states that a public manager allegedly influenced a land-use decision that would benefit an undisclosed personal investment.,Conflict of Interest,synthetic-conflict-of-interest-15,English,Turkana\nMsimamizi wa umma anadaiwa kushawishi uamuzi wa matumizi ya ardhi ambao ungenufaisha uwekezaji binafsi ambao hakufichua.,Conflict of Interest,synthetic-conflict-of-interest-15,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, msimamizi wa umma anadaiwa kushawishi uamuzi wa matumizi ya ardhi ambao ungenufaisha uwekezaji binafsi ambao hakufichua.",Conflict of Interest,synthetic-conflict-of-interest-15,Kiswahili,Turkana\n"A residents\' group says a public office repeatedly failed to publish meeting records required for community oversight, without alleging a payment or personal benefit.",Other,synthetic-other-01,English,Nairobi\n"The report states that a residents\' group says a public office repeatedly failed to publish meeting records required for community oversight, without alleging a payment or personal benefit.",Other,synthetic-other-01,English,Nairobi\n"Kikundi cha wakazi kinasema ofisi ya umma ilishindwa mara kwa mara kuchapisha kumbukumbu za mikutano zinazohitajika kwa ufuatiliaji wa jamii, bila kudai malipo au manufaa binafsi.",Other,synthetic-other-01,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, kikundi cha wakazi kinasema ofisi ya umma ilishindwa mara kwa mara kuchapisha kumbukumbu za mikutano zinazohitajika kwa ufuatiliaji wa jamii, bila kudai malipo au manufaa binafsi.",Other,synthetic-other-01,Kiswahili,Nairobi\n"A community committee alleges that a public department ignored required notice and consultation steps for a local plan, with no specific payment, threat, or favored person described.",Other,synthetic-other-02,English,Mombasa\n"The report states that a community committee alleges that a public department ignored required notice and consultation steps for a local plan, with no specific payment, threat, or favored person described.",Other,synthetic-other-02,English,Mombasa\n"Kamati ya jamii inadai kuwa idara ya umma ilipuuza hatua zinazohitajika za notisi na mashauriano kwa mpango wa eneo, bila kueleza malipo, tishio au mtu aliyependelewa.",Other,synthetic-other-02,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, kamati ya jamii inadai kuwa idara ya umma ilipuuza hatua zinazohitajika za notisi na mashauriano kwa mpango wa eneo, bila kueleza malipo, tishio au mtu aliyependelewa.",Other,synthetic-other-02,Kiswahili,Mombasa\n"A resident says an office did not provide a public spending explanation after repeated formal requests; the report describes no diversion, false record, or private gain.",Other,synthetic-other-03,English,Kisumu\n"The report states that a resident says an office did not provide a public spending explanation after repeated formal requests; the report describes no diversion, false record, or private gain.",Other,synthetic-other-03,English,Kisumu\n"Mkazi anasema ofisi haikutoa maelezo ya matumizi ya umma baada ya maombi rasmi kurudiwa; ripoti haielezi ubadhirifu, rekodi ya uongo au faida binafsi.",Other,synthetic-other-03,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, mkazi anasema ofisi haikutoa maelezo ya matumizi ya umma baada ya maombi rasmi kurudiwa; ripoti haielezi ubadhirifu, rekodi ya uongo au faida binafsi.",Other,synthetic-other-03,Kiswahili,Kisumu\n"A community group alleges that an oversight meeting was cancelled without explanation and minutes were not made available, without identifying a bribe or altered document.",Other,synthetic-other-04,English,Nakuru\n"The report states that a community group alleges that an oversight meeting was cancelled without explanation and minutes were not made available, without identifying a bribe or altered document.",Other,synthetic-other-04,English,Nakuru\n"Kikundi cha jamii kinadai kuwa mkutano wa uangalizi ulifutwa bila maelezo na kumbukumbu hazikupatikana, bila kutaja hongo au hati iliyobadilishwa.",Other,synthetic-other-04,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, kikundi cha jamii kinadai kuwa mkutano wa uangalizi ulifutwa bila maelezo na kumbukumbu hazikupatikana, bila kutaja hongo au hati iliyobadilishwa.",Other,synthetic-other-04,Kiswahili,Nakuru\n"A public notice about a local works plan was not displayed as required, according to residents; no contract manipulation or payment demand is alleged.",Other,synthetic-other-05,English,Turkana\n"The report states that a public notice about a local works plan was not displayed as required, according to residents; no contract manipulation or payment demand is alleged.",Other,synthetic-other-05,English,Turkana\n"Tangazo la umma kuhusu mpango wa kazi za eneo halikuwekwa kama ilivyohitajika, kulingana na wakazi; hakuna udanganyifu wa mkataba au ombi la malipo linalodaiwa.",Other,synthetic-other-05,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, tangazo la umma kuhusu mpango wa kazi za eneo halikuwekwa kama ilivyohitajika, kulingana na wakazi; hakuna udanganyifu wa mkataba au ombi la malipo linalodaiwa.",Other,synthetic-other-05,Kiswahili,Turkana\n"A committee reports that required attendance and decision records were not maintained for routine public meetings, without alleging forged entries.",Other,synthetic-other-06,English,Garissa\n"The report states that a committee reports that required attendance and decision records were not maintained for routine public meetings, without alleging forged entries.",Other,synthetic-other-06,English,Garissa\n"Kamati inaripoti kuwa rekodi zinazohitajika za mahudhurio na maamuzi hazikutunzwa kwa mikutano ya kawaida ya umma, bila kudai maingizo ya kughushi.",Other,synthetic-other-06,Kiswahili,Garissa\n"Kwa mujibu wa taarifa, kamati inaripoti kuwa rekodi zinazohitajika za mahudhurio na maamuzi hazikutunzwa kwa mikutano ya kawaida ya umma, bila kudai maingizo ya kughushi.",Other,synthetic-other-06,Kiswahili,Garissa\nResidents allege that a public feedback process was not held before a service plan was adopted; no personal beneficiary is identified.,Other,synthetic-other-07,English,Kilifi\nThe report states that residents allege that a public feedback process was not held before a service plan was adopted; no personal beneficiary is identified.,Other,synthetic-other-07,English,Kilifi\nWakazi wanadai kuwa mchakato wa maoni ya umma haukufanyika kabla ya mpango wa huduma kupitishwa; hakuna mnufaika binafsi aliyetajwa.,Other,synthetic-other-07,Kiswahili,Kilifi\n"Kwa mujibu wa taarifa, wakazi wanadai kuwa mchakato wa maoni ya umma haukufanyika kabla ya mpango wa huduma kupitishwa; hakuna mnufaika binafsi aliyetajwa.",Other,synthetic-other-07,Kiswahili,Kilifi\n"A civic group says a public office did not respond to a required audit follow-up, without describing missing funds or a personal payment.",Other,synthetic-other-08,English,Bungoma\n"The report states that a civic group says a public office did not respond to a required audit follow-up, without describing missing funds or a personal payment.",Other,synthetic-other-08,English,Bungoma\n"Kikundi cha kiraia kinasema ofisi ya umma haikujibu ufuatiliaji wa ukaguzi uliotakiwa, bila kueleza pesa zilizopotea au malipo binafsi.",Other,synthetic-other-08,Kiswahili,Bungoma\n"Kwa mujibu wa taarifa, kikundi cha kiraia kinasema ofisi ya umma haikujibu ufuatiliaji wa ukaguzi uliotakiwa, bila kueleza pesa zilizopotea au malipo binafsi.",Other,synthetic-other-08,Kiswahili,Bungoma\nA community alleges that a public service schedule was not posted in the required place and residents could not verify the announced timetable.,Other,synthetic-other-09,English,Meru\nThe report states that a community alleges that a public service schedule was not posted in the required place and residents could not verify the announced timetable.,Other,synthetic-other-09,English,Meru\nJamii inadai kuwa ratiba ya huduma ya umma haikuwekwa mahali palipotakiwa na wakazi hawakuweza kuthibitisha muda uliotangazwa.,Other,synthetic-other-09,Kiswahili,Meru\n"Kwa mujibu wa taarifa, jamii inadai kuwa ratiba ya huduma ya umma haikuwekwa mahali palipotakiwa na wakazi hawakuweza kuthibitisha muda uliotangazwa.",Other,synthetic-other-09,Kiswahili,Meru\n"A residents\' association says a public department did not provide a required explanation of how it selected service priorities, with no allegation of a private interest.",Other,synthetic-other-10,English,Kitui\n"The report states that a residents\' association says a public department did not provide a required explanation of how it selected service priorities, with no allegation of a private interest.",Other,synthetic-other-10,English,Kitui\n"Chama cha wakazi kinasema idara ya umma haikutoa maelezo yaliyotakiwa kuhusu jinsi ilivyochagua vipaumbele vya huduma, bila madai ya maslahi binafsi.",Other,synthetic-other-10,Kiswahili,Kitui\n"Kwa mujibu wa taarifa, chama cha wakazi kinasema idara ya umma haikutoa maelezo yaliyotakiwa kuhusu jinsi ilivyochagua vipaumbele vya huduma, bila madai ya maslahi binafsi.",Other,synthetic-other-10,Kiswahili,Kitui\n"A community group alleges that a public consultation notice omitted the date and location, preventing residents from taking part; no bribe or threat is described.",Other,synthetic-other-11,English,Nairobi\n"The report states that a community group alleges that a public consultation notice omitted the date and location, preventing residents from taking part; no bribe or threat is described.",Other,synthetic-other-11,English,Nairobi\n"Kikundi cha jamii kinadai kuwa notisi ya mashauriano ya umma iliacha tarehe na mahali, na kuwazuia wakazi kushiriki; hakuna hongo au tishio lililoelezwa.",Other,synthetic-other-11,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, kikundi cha jamii kinadai kuwa notisi ya mashauriano ya umma iliacha tarehe na mahali, na kuwazuia wakazi kushiriki; hakuna hongo au tishio lililoelezwa.",Other,synthetic-other-11,Kiswahili,Nairobi\n"A local committee says quarterly public-service statistics were not released as required, without alleging that figures were falsified.",Other,synthetic-other-12,English,Mombasa\n"The report states that a local committee says quarterly public-service statistics were not released as required, without alleging that figures were falsified.",Other,synthetic-other-12,English,Mombasa\n"Kamati ya eneo inasema takwimu za huduma za umma za robo mwaka hazikutolewa kama ilivyotakiwa, bila kudai kuwa takwimu zilighushiwa.",Other,synthetic-other-12,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, kamati ya eneo inasema takwimu za huduma za umma za robo mwaka hazikutolewa kama ilivyotakiwa, bila kudai kuwa takwimu zilighushiwa.",Other,synthetic-other-12,Kiswahili,Mombasa\n"Residents report that a public office did not keep a required register of service requests, with no specific official benefit alleged.",Other,synthetic-other-13,English,Kisumu\n"The report states that residents report that a public office did not keep a required register of service requests, with no specific official benefit alleged.",Other,synthetic-other-13,English,Kisumu\n"Wakazi wanaripoti kuwa ofisi ya umma haikutunza daftari lililohitajika la maombi ya huduma, bila kudai manufaa maalum ya afisa.",Other,synthetic-other-13,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, wakazi wanaripoti kuwa ofisi ya umma haikutunza daftari lililohitajika la maombi ya huduma, bila kudai manufaa maalum ya afisa.",Other,synthetic-other-13,Kiswahili,Kisumu\nA civic group says the public office failed to disclose the criteria used to set service priorities; the report alleges no tender or payment scheme.,Other,synthetic-other-14,English,Nakuru\nThe report states that a civic group says the public office failed to disclose the criteria used to set service priorities; the report alleges no tender or payment scheme.,Other,synthetic-other-14,English,Nakuru\nKikundi cha kiraia kinasema ofisi ya umma haikufichua vigezo vya kupanga vipaumbele vya huduma; ripoti haina madai ya zabuni au mpango wa malipo.,Other,synthetic-other-14,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, kikundi cha kiraia kinasema ofisi ya umma haikufichua vigezo vya kupanga vipaumbele vya huduma; ripoti haina madai ya zabuni au mpango wa malipo.",Other,synthetic-other-14,Kiswahili,Nakuru\n"A community alleges that a required public notice about a service change was not issued, without describing a threat, altered record, or personal gain.",Other,synthetic-other-15,English,Turkana\n"The report states that a community alleges that a required public notice about a service change was not issued, without describing a threat, altered record, or personal gain.",Other,synthetic-other-15,English,Turkana\n"Jamii inadai kuwa notisi ya umma iliyohitajika kuhusu mabadiliko ya huduma haikutolewa, bila kueleza tishio, rekodi iliyobadilishwa au faida binafsi.",Other,synthetic-other-15,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, jamii inadai kuwa notisi ya umma iliyohitajika kuhusu mabadiliko ya huduma haikutolewa, bila kueleza tishio, rekodi iliyobadilishwa au faida binafsi.",Other,synthetic-other-15,Kiswahili,Turkana'   # repr() keeps quoting/commas safe

df = pd.read_csv(io.StringIO(CSV_DATA), dtype=str, keep_default_na=False)
df = df.apply(lambda col: col.str.strip())

# --- Dataset audit gates (mirrors ml/src/dataset.py::validate_for_training) ---
counts = Counter(df["label"])
groups = {c: df[df["label"] == c]["group_id"].nunique() for c in CATEGORIES}
duplicates = len(df) - len({t.strip().casefold() for t in df["text"]})

assert set(counts) == set(CATEGORIES), f"missing categories: {set(CATEGORIES) - set(counts)}"
assert duplicates == 0, f"exact duplicate text rows: {duplicates}"
assert all(groups[c] >= 10 for c in CATEGORIES), "each category needs >=10 incident groups"

print(f"Rows               : {len(df)}")
print(f"Distinct incidents : {df['group_id'].nunique()}")
print(f"Rows per category  : {dict(sorted(counts.items()))}")
print(f"Duplicate texts    : {duplicates}")


## 2.3 Exploratory Data Analysis

*(Rubric: 2 marks — key insights, supported by visuals/statistics)*


In [ ]:
df["word_count"] = df["text"].str.split().str.len()

print("Shape:", df.shape)
print("\nClass distribution:")
print(df["label"].value_counts().to_string())
print("\nLanguage mix:")
print(df["language"].value_counts().to_string())
print("\nNarrative length in words:")
print(df["word_count"].describe().round(1).to_string())
print("\nIncident groups per category:")
print(pd.Series(groups).to_string())


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# (a) Class balance
counts_sorted = df["label"].value_counts().sort_values()
axes[0, 0].barh(counts_sorted.index, counts_sorted.values, color="#0d9488")
axes[0, 0].set_title("(a) Rows per category")
axes[0, 0].set_xlabel("rows")

# (b) Narratives per incident -> shows the grouping structure
per_group = df.groupby("group_id").size()
axes[0, 1].hist(per_group.values, bins=range(1, per_group.max() + 2), color="#0f766e",
                edgecolor="white", align="left")
axes[0, 1].set_title("(b) Variants per incident group")
axes[0, 1].set_xlabel("rows describing the same incident")

# (c) Narrative length by language
for lang, colour in (("English", "#0f766e"), ("Kiswahili", "#f59e0b")):
    axes[1, 0].hist(df[df["language"] == lang]["word_count"], bins=12, alpha=0.75,
                    label=lang, color=colour)
axes[1, 0].set_title("(c) Narrative length by language")
axes[1, 0].set_xlabel("words per report")
axes[1, 0].legend()

# (d) Vocabulary overlap: shared vs unique words per category
from sklearn.feature_extraction.text import CountVectorizer
vec = CountVectorizer(lowercase=True)
vocab_by_cat = {}
for cat in CATEGORIES:
    vec.fit(df[df["label"] == cat]["text"])
    vocab_by_cat[cat] = set(vec.vocabulary_)
shared = set.intersection(*vocab_by_cat.values())
axes[1, 1].barh(list(vocab_by_cat), [len(vocab_by_cat[c] - shared) for c in vocab_by_cat],
                color="#0d9488")
axes[1, 1].set_title(f"(d) Category-distinctive words (shared vocabulary = {len(shared)})")
axes[1, 1].set_xlabel("unique terms")

plt.tight_layout()
plt.show()


### Insights that shaped the modelling decisions

1. **Balanced classes** (60 rows per category) — a property of the *designed* corpus, not of real
   reports. Real corruption reporting will be heavily imbalanced, which is why the pipeline uses
   `class_weight="balanced"` and reports **macro F1** rather than plain accuracy.
2. **Narratives are not independent**: each incident appears as 4 paraphrases. Treating rows as
   independent would let the model memorise an incident in training and be scored on a paraphrase
   of it — textbook leakage. Validation therefore **groups by incident**.
3. **Short narratives** (~20–30 words) — favour sparse TF-IDF over long-context models.
4. **Two languages** with different morphology — Kiswahili is agglutinative, so word tokens
   fragment; character n-grams recover much of that signal.
5. **Distinctive vs shared vocabulary** (panel d) — categories do have discriminative vocabulary,
   but share a large common core (words like *report*, *officer*, *payment*), which is exactly why a
   simple keyword classifier alone cannot separate them.

## 2.4 Preprocessing and feature engineering

Preprocessing is embedded **inside** the model pipeline rather than applied to the whole dataset up
front. This is deliberate: vectorisers are then fitted on training folds only, so no information
from the held-out fold leaks into the features.

| Step | Setting / purpose |
|---|---|
| Case folding + Unicode accent stripping | Normalise surface variation |
| Whitespace collapse | Remove formatting noise |
| **Word TF-IDF, n-grams (1–2), sublinear TF** | Captures phrases like *"bid rigging"*; sublinear TF dampens long repeated narratives |
| **Character TF-IDF, `char_wb`, n-grams (3–5)** | Robust to inflection, compounding, misspelling, Kiswahili morphology |
| `min_df=2` | Drops singleton terms (noise on small corpora) |
| `max_features=50 000` per vectoriser | Bounds dimensionality |
| `class_weight="balanced"` | Prepares for class imbalance in real data |
| **No stop-word removal** | Words like *not*, *without*, *before* carry meaning in allegations ("paid **without** receipt") |


---
# 3. Methodology and Model Selection
*(Rubric: 4 marks model selection & justification · 3 marks technical implementation)*

## 3.1 Model candidates and justification

Two **pre-declared** candidates are compared — not an open-ended hyperparameter search. On a
synthetic corpus an exhaustive search would inflate results without adding evidence about real data.

| Candidate | Why it is included | Known weakness |
|---|---|---|
| **Word TF-IDF (1–2) + Logistic Regression** (`balanced`) | Strong, interpretable text baseline; coefficient signs show which words drive a label | Ignores morphology; brittle to misspelling |
| **Word + char TF-IDF → Linear SVM** (`balanced`) | Character n-grams absorb spelling and Kiswahili agglutination; linear SVM handles very high-dimensional sparse input efficiently | Decision scores are less interpretable than probabilities |
| *(considered, rejected)* fine-tuned transformer (e.g. mBERT) | Potentially strongest on real multilingual text | No real labelled data to fine-tune on; 540 synthetic rows would demonstrate only over-fitting; too heavy for the deployment target |

**Why linear models on TF-IDF are the right choice here:** the input is short, domain-specific
narrative text; the corpus is small; interpretability matters for a governance-sensitive platform;
and the model must be cheap enough to run on submission without noticeable latency.

## 3.2 Validation design — the decisive methodological choice

```
OUTER : 5-fold StratifiedGroupKFold   → estimates end-to-end generalisation
INNER : 3-fold stratified GROUP CV    → selects between the two candidates
METRIC: macro F1                      → every category weighs equally
GROUP : incident group_id             → paraphrases of one incident never straddle a fold
```

Three properties make this defensible:

1. **Grouped folds** — prevents paraphrase leakage (§2.3 insight 2).
2. **Nested selection** — model choice happens *inside* each outer training fold, so the reported
   score includes the cost of choosing a model. Reporting the score of a model chosen on all the
   data would be optimistically biased.
3. **Trivial baseline alongside** — a fold-local majority-class predictor is scored on the same
   folds, so every result is interpreted relative to what a do-nothing strategy achieves.

## 3.3 Hyperparameters (fixed a priori)

| Component | Setting | Rationale |
|---|---|---|
| Word TF-IDF | `ngram_range=(1,2)`, `min_df=2`, `max_features=50000`, `sublinear_tf=True` | Phrases + noise control |
| Char TF-IDF | `analyzer="char_wb"`, `ngram_range=(3,5)`, `min_df=2`, `max_features=50000` | Multilingual morphology |
| Logistic Regression | `class_weight="balanced"`, `max_iter=2000`, `random_state=42` | Imbalance-aware, reproducible |
| Linear SVM | `class_weight="balanced"`, `random_state=42` | High-dimensional sparse strength |
| Outer / inner folds | 5 / 3, shuffled, seeds 42 / 100+fold | Stable, pre-declared |


## 3.4 Implementation — model construction


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

MODEL_CANDIDATES = ("word_logistic", "word_char_linear_svc")


def _word_features():
    return TfidfVectorizer(
        analyzer="word", lowercase=True, strip_accents="unicode",
        ngram_range=(1, 2), min_df=2, max_features=50000, sublinear_tf=True,
    )


def _char_features():
    return TfidfVectorizer(
        analyzer="char_wb", lowercase=True, strip_accents="unicode",
        ngram_range=(3, 5), min_df=2, max_features=50000, sublinear_tf=True,
    )


def build_model(name):
    if name == "word_logistic":
        return Pipeline([
            ("features", _word_features()),
            ("classifier", LogisticRegression(class_weight="balanced", max_iter=2000,
                                              random_state=42)),
        ])
    if name == "word_char_linear_svc":
        return Pipeline([
            ("features", FeatureUnion([("word", _word_features()), ("char", _char_features())])),
            ("classifier", LinearSVC(class_weight="balanced", random_state=42)),
        ])
    raise ValueError(f"unknown candidate: {name}")


# Sanity check: both pipelines build and fit on a stratified sample covering all 9 classes.
_sample = df.groupby("label").head(6)   # 54 rows, every category represented
for _name in MODEL_CANDIDATES:
    _model = build_model(_name).fit(_sample["text"], _sample["label"])
    _accuracy = (_model.predict(_sample["text"]) == _sample["label"]).mean()
    print(f"{_name:26s} build+fit OK on {len(_sample)} rows (in-sample accuracy {_accuracy:.2f})")


## 3.5 Implementation — nested grouped cross-validation


In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             precision_recall_fscore_support, confusion_matrix)

TEXTS = df["text"].tolist()
LABELS = df["label"].tolist()
GROUPS = df["group_id"].tolist()


def compute_metrics(y_true, y_pred, labels=CATEGORIES):
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    return {
        "accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
        "balanced_accuracy": round(float(balanced_accuracy_score(y_true, y_pred)), 4),
        "macro_f1": round(float(f1_score(y_true, y_pred, labels=labels, average="macro",
                                         zero_division=0)), 4),
        "per_category": {
            c: {"precision": round(float(p[i]), 4), "recall": round(float(r[i]), 4),
                "f1": round(float(f[i]), 4), "support": int(s[i])}
            for i, c in enumerate(labels)
        },
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels).tolist(),
    }


def cross_validate_candidate(train_idx, name, n_splits, seed):
    """Grouped CV inside an outer training fold; returns out-of-fold predictions + macro F1."""
    t = [TEXTS[i] for i in train_idx]
    y = [LABELS[i] for i in train_idx]
    g = [GROUPS[i] for i in train_idx]
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    preds = [None] * len(train_idx)
    for tr, te in splitter.split(t, y, g):
        assert not (set(g[i] for i in tr) & set(g[i] for i in te)), "group leakage"
        model = build_model(name).fit([t[i] for i in tr], [y[i] for i in tr])
        for i, pred in zip(te, model.predict([t[i] for i in te])):
            preds[i] = str(pred)
    return preds, compute_metrics(y, preds)["macro_f1"]


def select_model(train_idx, n_splits=3, seed=42):
    scores = {name: cross_validate_candidate(train_idx, name, n_splits, seed)[1]
              for name in MODEL_CANDIDATES}
    return max(scores, key=scores.get), scores


print("Helpers ready.")


In [ ]:
outer_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = [None] * len(df)
baseline_preds = [None] * len(df)
fold_records = []

for fold, (tr, te) in enumerate(outer_splitter.split(TEXTS, LABELS, GROUPS), start=1):
    # Leakage guard: no incident on both sides of the split.
    assert not (set(GROUPS[i] for i in tr) & set(GROUPS[i] for i in te)), "group leakage"

    chosen, inner_scores = select_model(tr, n_splits=3, seed=100 + fold)

    model = build_model(chosen).fit([TEXTS[i] for i in tr], [LABELS[i] for i in tr])
    fold_pred = model.predict([TEXTS[i] for i in te])
    for i, pred in zip(te, fold_pred):
        oof_preds[i] = str(pred)

    majority = Counter(LABELS[i] for i in tr).most_common(1)[0][0]
    for i in te:
        baseline_preds[i] = majority

    fold_records.append({
        "fold": fold,
        "selected": chosen,
        "test_rows": len(te),
        "test_incidents": len({GROUPS[i] for i in te}),
        "macro_f1": round(float(f1_score([LABELS[i] for i in te], fold_pred,
                                         labels=CATEGORIES, average="macro",
                                         zero_division=0)), 4),
        **{f"inner_{k}": v for k, v in inner_scores.items()},
    })

folds_df = pd.DataFrame(fold_records)
assert all(p is not None for p in oof_preds), "every row needs an out-of-fold prediction"
print(folds_df.to_string(index=False))


---
# 4. Model Evaluation
*(Rubric: 4 marks metrics & interpretation · 2 marks comparative analysis · 2 marks visualisation)*

## 4.1 Metrics and why each is used

| Metric | What it means | Why it is here |
|---|---|---|
| **Accuracy** | % of reports assigned the correct category | Intuitive headline number |
| **Balanced accuracy** | Mean recall across categories | Insensitive to class sizes |
| **Macro F1** | Mean of per-category F1 scores | **Primary metric** — a rare category counts as much as a common one |
| **Per-category precision / recall / F1** | Which categories are reliable, which are confused | Operational: a missed *Extortion* case matters more than a missed *Other* |
| **Confusion matrix** | Where mistakes go | Reveals semantic boundaries the model confuses |


## 4.2 Headline results vs. trivial baseline


In [ ]:
headline = pd.DataFrame([
    {"model": "Fold-local majority baseline",
     "accuracy": compute_metrics(LABELS, baseline_preds)["accuracy"],
     "balanced_accuracy": compute_metrics(LABELS, baseline_preds)["balanced_accuracy"],
     "macro_f1": compute_metrics(LABELS, baseline_preds)["macro_f1"]},
    {"model": "Nested grouped CV (selected model)",
     **{k: v for k, v in compute_metrics(LABELS, oof_preds).items()
        if k in ("accuracy", "balanced_accuracy", "macro_f1")}},
])

print(headline.to_string(index=False))
print()
print(f"Out-of-fold macro F1 across outer folds: "
      f"mean {folds_df.macro_f1.mean():.4f}  (spread {folds_df.macro_f1.std(ddof=0):.4f})")
print(f"Models chosen per fold: {folds_df.selected.value_counts().to_dict()}")


## 4.3 Comparative analysis

Three comparisons are reported, which is what "benchmarking against baseline models or other
approaches" requires:

1. **Model vs trivial baseline** (table above). The majority classifier scores ≈1/9 ≈ 0.11 by
   chance on a balanced 9-class problem; the trained model must clear that by a wide margin to be
   worth deploying.
2. **Candidate vs candidate** — the per-fold inner scores (`inner_word_logistic` vs
   `inner_word_char_linear_svc`) show *which* model won during selection and by how much,
   rather than hiding the comparison inside a single final number.
3. **Category vs category** — per-category F1 (§4.5) shows whether errors concentrate in one
   semantic boundary (e.g. *Bribery* vs *Extortion*), which is the most actionable comparison for
   the review team.


## 4.4 Visual representation


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

headline.set_index("model")[["accuracy", "balanced_accuracy", "macro_f1"]].plot.bar(
    ax=axes[0], color=["#94a3b8", "#0d9488", "#0f766e"], rot=10)
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("score")
axes[0].set_title("(a) Trained model vs trivial baseline")
axes[0].legend(loc="lower right")

cm = compute_metrics(LABELS, oof_preds)["confusion_matrix"]
im = axes[1].imshow(cm, cmap="Blues")
axes[1].set_xticks(range(len(CATEGORIES)))
axes[1].set_yticks(range(len(CATEGORIES)))
axes[1].set_xticklabels([c[:12] for c in CATEGORIES], rotation=60, ha="right", fontsize=8)
axes[1].set_yticklabels([c[:16] for c in CATEGORIES], fontsize=8)
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")
axes[1].set_title("(b) Confusion matrix (out-of-fold)")
vmax = max(max(row) for row in cm)
for i in range(len(CATEGORIES)):
    for j in range(len(CATEGORIES)):
        if cm[i][j]:
            axes[1].text(j, i, cm[i][j], ha="center", va="center", fontsize=7,
                         color="white" if cm[i][j] > vmax / 2 else "black")
fig.colorbar(im, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.show()


## 4.5 Per-category performance and its operational meaning


In [ ]:
per_cat = pd.DataFrame(compute_metrics(LABELS, oof_preds)["per_category"]).T
per_cat.columns = ["precision", "recall", "F1", "support"]
per_cat = per_cat.sort_values("F1")
print(per_cat.to_string())

per_cat["F1"].plot(kind="barh", figsize=(9, 4.2), color="#0d9488",
                   title="Per-category F1 (out-of-fold predictions)")
plt.xlabel("F1")
plt.xlim(0, 1.0)
plt.tight_layout()
plt.show()


**Operational reading:** these synthetic scores do not establish a safe confidence floor. The live
demo shows only an advisory category suggestion, never a probability or confidence value, and the
double-blind human review decides the label. The model never auto-labels a report.


## 4.6 Language subgroup analysis (an exploratory fairness check)

Kenya Watch is multilingual by design, so metrics are computed per language. This is an
**exploratory** look, not a fairness guarantee — and the Kiswahili in this corpus is draft text that
has not been reviewed by a native speaker, so a gap here is a signal to obtain proper review, not a
conclusion.


In [ ]:
subgroup_rows = []
for language in sorted(df["language"].unique()):
    idx = df.index[df["language"] == language].tolist()
    present = tuple(c for c in CATEGORIES if c in {LABELS[i] for i in idx})
    m = compute_metrics([LABELS[i] for i in idx], [oof_preds[i] for i in idx], present)
    subgroup_rows.append({"language": language, "rows": len(idx),
                          "accuracy": m["accuracy"], "macro_f1": m["macro_f1"]})
print(pd.DataFrame(subgroup_rows).to_string(index=False))


## 4.7 What these numbers do **not** prove

Stated plainly, because this matters more than any score:

1. **The data is synthetic.** It was written to be learnable: templatic phrasing and balanced classes
   inflate scores. These numbers say nothing about real Kenyan corruption reports.
2. **No temporal holdout.** Real deployment needs a later-in-time, incident-separated test set to
   detect drift as language and reporting patterns change.
3. **No native-speaker validation** of Kiswahili, and no code-switching (English/Swahili mixed within
   one narrative), which is very common in real reports.
4. **No probability calibration.** The current model's decision margins are not probabilities and
   are not exposed as confidence values in the live application.
5. **Fairness is unexplored** beyond a language split; proper evaluation needs stakeholder-defined
   acceptable error rates per category.

> **The defensible claim concerns the pipeline, not the score:** "I built and validated a
> leakage-resistant, nested, multilingual text-classification pipeline, and I can explain precisely
> why its current numbers cannot be presented as real-world accuracy."


---
# 5. Deployment
*(Rubric: 2 marks deployment strategy · 6 marks demonstration)*

## 5.1 Where this sits in the live system

```
Citizen submits ──► POST /api/reports ──► report saved ──► response returned immediately
                                                │
                                                └─► setImmediate(...)   [asynchronous — never blocks]
                                                        │
                                              classifyInBackground()
                                                        │
                             backend/utils/classifier.js
                                         │
                              persistent Python worker
                              ┌──────────┴──────────────┐
                              │                         │
                  synthetic-trained ML category    keyword fallback
                  + model version/source            if worker unavailable
```

The live integration is demonstrative only: the trained category suggestion comes from fictional
synthetic examples, is not a probability or a case finding, and is disclosed to reviewers only after
they submit an independent category assessment. Urgency and tags remain keyword-based. If the
artifact or Python worker is unavailable, submission still succeeds and the system records that it
used the keyword category fallback.

## 5.2 Rollout plan (governance-gated)

| Step | Requirement | Status |
|---|---|---|
| 1 | Collect adjudicated labels via double-blind review (`categoryReview`) | **Live workflow exists** |
| 2 | Export approved dataset (`ml/scripts/export-reviewed-dataset.js`, allowlist + approval) | **Implemented** |
| 3 | De-identify and verify narratives manually | Documented process |
| 4 | Re-run this pipeline on real data | **This notebook / `ml/src/train.py`** |
| 5 | Freeze a later-time, incident-separated holdout | Future |
| 6 | Compare model predictions with independent human decisions on approved real data | Future |
| 7 | Consider operational use only after real-world validation and governance review | Future |

## 5.3 Serving options when approved

The artifact is a `joblib`-serialised scikit-learn `Pipeline`, so integration is straightforward and
already isolated behind one function (`classifyInBackground` in `backend/routes/report.routes.js`):

- **(a) Python sidecar** — a small FastAPI/Flask service exposing `POST /classify`, called by Express.
- **(b) Export to ONNX** — run natively from Node without a Python runtime.
- **(c) Batch pre-computation** — classify on a queue worker; no request-path dependency at all.

Option (c) is the lowest-risk first step because it keeps the public submission path completely
independent of the model.

## 5.4 Demonstration (in-notebook)

Below: fit the final model on all data, then classify **unseen** sentences — including one Kiswahili
example and one deliberately ambiguous report — to show what the reviewer-facing suggestion looks
like, and to show that the model's margin is low when the text is ambiguous.


In [ ]:
final_model_name, final_scores = select_model(list(range(len(df))), n_splits=5, seed=42)
final_model = build_model(final_model_name).fit(TEXTS, LABELS)

print(f"Selected candidate : {final_model_name}")
print(f"Inner macro F1     : { {k: round(v, 4) for k, v in final_scores.items()} }")
print()

demonstration_samples = [
    # Clear example: an unofficial payment for a permit
    "An officer at the county offices told me to pay 10,000 shillings in cash before he would "
    "sign my business permit.",
    # Clear example: funds unaccounted for
    "The money collected from market stalls every week does not appear in the ward development "
    "account and nobody can explain where it went.",
    # Clear example: tender manipulation
    "The same company won three tenders in one month and all of them were evaluated by the same "
    "committee member.",
    # Kiswahili
    "Afisa aliniomba kitu kidogo kabla ya kunipatia kibali cha biashara yangu.",
    # Ambiguous / insufficient information
    "I have concerns about how things are handled at the office but I do not have details.",
]

def suggest(text, model, top_k=2):
    """Ranked (category, score) suggestions - how a reviewer would see it."""
    if hasattr(model, "decision_function"):
        scores = model.decision_function([text])[0]
        order = scores.argsort()[::-1][:top_k]
        return [(model.classes_[i], float(scores[i])) for i in order]
    return [(model.predict([text])[0], float("nan"))]


print(f"{'SUGGESTION':34s} {'RUNNER-UP':28s} SAMPLE")
print("-" * 118)
for sample in demonstration_samples:
    ranked = suggest(sample, final_model)
    first = f"{ranked[0][0]} ({ranked[0][1]:+.2f})"
    second = f"{ranked[1][0]} ({ranked[1][1]:+.2f})" if len(ranked) > 1 else ""
    print(f"{first:34s} {second:28s} {sample[:58]}")

print()
print("Interpretation")
print("-------------")
print("The suggestion list is advisory. Because the model is a LinearSVC, raw scores are")
print("ranked margins, not probabilities. They are not shown as confidence in the live app.")
print("Where categories overlap, the model's suggestion is only a starting point; the")
print("reviewer's independent judgment decides.")


### How the same triage concept appears in the running application

Walk through these three touchpoints during the live demo:

1. **Reviewer Queue** — after submitting an independent category assessment, reviewers see the
   model's advisory category, its synthetic-data warning and model version. The urgency badge is
   produced by keyword rules; the queue is sorted urgency-first, then by classification age.
2. **Admin overview → Escalations** — flags `High`/`Critical` cases still unclaimed past the
   threshold, timed from *classification completion*, not from submission.
3. **Public statistics** — a category × urgency heatmap (aggregate only) so the public can see where
   triage pressure is concentrated, without exposing any individual case.

### Persisting the artifact


In [ ]:
import hashlib
import joblib

# In Colab this writes to /content/artifacts; in the repository it mirrors ml/artifacts/.
artifact_dir = Path("artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)
model_path = artifact_dir / "category_model.joblib"
joblib.dump(final_model, model_path)
artifact_sha256 = hashlib.sha256(model_path.read_bytes()).hexdigest()
(artifact_dir / "model_metadata.json").write_text(json.dumps({
    "model": final_model_name,
    "model_version": artifact_sha256[:16],
    "artifact_sha256": artifact_sha256,
    "categories": list(CATEGORIES),
    "training_rows": len(df),
    "training_group_count": len(set(GROUPS)),
    "data_status": "synthetic_only_not_for_real_world_validation_or_production_training",
    "real_world_performance_claim": False,
    "note": "Synthetic-only pipeline demonstration; not valid for real-world validation or production.",
}, indent=2), encoding="utf-8")

print("Artifact written to:", artifact_dir.resolve())
print("Files:", sorted(p.name for p in artifact_dir.glob("*.joblib")))
print("NOTE: this synthetic-only artifact is experimental and not validated for real reports.")


---
# Summary

| Rubric area | Delivered in this notebook |
|---|---|
| **Introduction & problem** (5) | Corruption reporting fails on fear, feedback and triage; ML assists triage only, and never sees reporter identity |
| **Dataset & preprocessing** (9) | 540-row, 9-category, incident-grouped synthetic corpus; audit gates; 4-panel EDA with five insights; learned in-pipeline TF-IDF features |
| **Methodology** (7) | Two pre-declared candidates with justification; nested 5×3 StratifiedGroupKFold; fixed hyperparameters with rationale; leakage assertions |
| **Evaluation** (8) | Accuracy, balanced accuracy, macro F1, per-category P/R/F1, confusion matrix, majority baseline comparison, candidate comparison, language subgroups, explicit limitations |
| **Deployment** (8) | Three-way integration strategy with an isolated hook point, governance-gated rollout plan, in-notebook demonstration of unseen predictions |

### Final honest statement

> The contribution here is a **rigorous, leakage-resistant, multilingual text-classification
> pipeline** with a documented evaluation and deployment plan, together with a live platform in
> which the ML layer is deliberately constrained to be assistive and auditable. The synthetic
> scores are a *pipeline sanity check* — not a claim about real-world accuracy.
